# 00 — Data Preparation (all cases)

**Purpose.** Turn every raw Muse 2 `.xdf` recording in `01. Data/00. Raw/` into a set of
tidy, model-ready CSV tables in `01. Data/01. Imported/` — one CSV per case.

**One row of the output = one accepted 2-second epoch**, described by 20 Welch
band-power features (4 EEG channels x 5 frequency bands) plus the labels needed by the
downstream analysis notebooks (subject, session, case, marker, block, age, gender).

---

## Pipeline

```
 01. Data/00. Raw/sub-PXXX/ses-SXXX/eeg/*task-CaseN*.xdf         34 recordings
        |
 [1] discover_recordings()  walk the folder tree; read subject / session / case
        |                   from the folder names, warn on filename mismatches
        |
        |   ┌─────────────── steps 2-4 run once per recording, ───────────────┐
        |   │                in parallel across all CPU cores (N_JOBS)        │
        v   v                                                                 │
 [2] load_xdf()             pyxdf -> DataFrame; merge_asof stamps every EEG   │
        |                   sample with the marker in force; trim the lead-in │
        |                                                                     │
 [3] preprocess()           1. Muse sentinel values -> NaN                    │
        |                   2. KNN interpolation (fills gaps for the filters)  │
        |                   3. Butterworth bandpass 1-50 Hz, zero-phase        │
        |                   4. IIR notch at 50 Hz (mains hum)                  │
        |                   5. |x| > 150 uV -> NaN -> KNN interpolation        │
        |                   6. Linked-mastoid re-reference: subtract           │
        |                      mean(TP9, TP10) from every channel              │
        |                                                                      │
 [4] extract_features()     cut into 2 s epochs inside each contiguous marker  │
        |                   block -> drop epochs with peak-to-peak > 420 uV    │
        |                   -> welch_psd() -> integrate power in each band     │
        |   └──────────────────────────────────────────────────────────────────┘
        v
 [5] load_person_metadata() Age + Gender per subject, from Respondents_profiles.csv
        |
 [6] (dispatch + collect)   gather every worker's feature table; report failures
        |
        |--- [7] time-domain check      raw vs cleaned signal against time
        |--- [8] frequency-domain check Welch periodograms + feature vectors
        |                                    -> reports/figures/preprocessing/
        v
 [9] assemble + save        join metadata, order columns, split by case
        |
 [10] verify                marker / subject / value-range checks on the result
        v
 01. Data/01. Imported/Case1_epochs.csv ... Case5_epochs.csv
```

Steps 7 and 8 are diagnostics — they draw figures and change no data. Everything the
CSV contains is produced by steps 2–4.

## The 5 cases

| Case | Protocol | Markers used in that protocol |
|---|---|---|
| 1 | AI vs. human coding | 1 = eyes open, 2 = eyes closed, 10 = watch sport, 99 = rest, 20 = code solo, 30 = code with AI |
| 2 | Person identification | 1 = eyes open, 2 = eyes closed, 10 = social media |
| 3 | Activity classification | 1, 2, 10 = game, 20 = shop, 30 = market, 40 = IQ test, 99 = rest |
| 4 | Sex differences (resting state) | 1 = eyes open, 2 = eyes closed, 10 = low load (breath counting) |
| 5 | Attention span | 1, 2, 11 = short instructional, 12 = long instructional, 21 = short entertainment, 22 = long entertainment, 99 = rest |

Markers are **kept as-is** here. Filtering (for example dropping the rest blocks, marker 99)
is the job of each analysis notebook, not of data preparation.

---

**Everything you can tune lives in the single `[SETTINGS]` cell below.**
Change a value there and re-run the whole notebook (Kernel -> Restart & Run All).

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [SETTINGS]  —  the only cell you need to edit
# ══════════════════════════════════════════════════════════════════════════════
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
# All paths are relative to the project root (the folder holding "01. Data").
# The next cell resolves the project root automatically, so these stay short.
RAW_DIR       = Path("01. Data") / "00. Raw"        # input:  sub-PXXX/ses-SXXX/eeg/*.xdf
OUT_DIR       = Path("01. Data") / "01. Imported"   # output: one CSV per case
PROFILES_CSV  = Path("01. Data") / "Respondents_profiles.csv"   # demographics table

# ── Which recordings to process ───────────────────────────────────────────────
# Cases to include. Every case gets its own CSV file in OUT_DIR.
# Set to e.g. [2] to re-run only Case 2.
CASES = [1, 2, 3, 4, 5]

# Filename pattern searched inside each  sub-*/ses-*/eeg/  folder.
# "{case}" is substituted with "Case1", "Case2", ... in turn.
FILE_GLOB = "*task-{case}_run-*_eeg.xdf"

# ── EEG acquisition (Muse 2 hardware facts — do not change unless the device does)
CH_COLS   = ["TP9", "AF7", "AF8", "TP10"]   # the 4 EEG electrodes; "Right AUX" is dropped
SFREQ     = 256.0                            # sample rate in Hz
SENTINELS = [-1000.0, 999.511719]            # magic values the Muse firmware writes for
                                             # dropped / invalid samples (in uV)

# ── Stage 3: filtering ────────────────────────────────────────────────────────
BANDPASS_LO    = 1.0    # Hz — high-pass corner: removes DC offset and slow drift
BANDPASS_HI    = 50.0   # Hz — low-pass corner: sits above the gamma band we extract (44 Hz)
BANDPASS_ORDER = 4      # Butterworth order; zero-phase filtering effectively doubles it
LINE_FREQ      = 50     # Hz — mains hum to notch out (use 60 in North America)
NOTCH_Q        = 30     # notch quality factor: higher = narrower notch

# ── Stage 3: artefact handling ────────────────────────────────────────────────
THRESHOLD = 150   # uV — any sample beyond +/-THRESHOLD is treated as an artefact,
                  # blanked to NaN and re-filled by KNN interpolation.
                  # Rationale: ~3 sigma of the noisiest channel after filtering.
KNN_K     = 5     # how many nearest valid neighbours the interpolation averages over

# ── Stage 4: epoching ─────────────────────────────────────────────────────────
EPOCH_S    = 2.0   # epoch (window) length in seconds -> 512 samples at 256 Hz
PTP_THRESH = 420   # uV — reject the whole epoch if any channel's peak-to-peak
                   # amplitude exceeds this. Catches artefacts that survived stage 3.

# Epoch inside *contiguous* marker blocks rather than pooling all samples that share
# a marker value. This matters because several protocols reuse a marker code: e.g.
# marker 1 (eyes open) runs at the start AND at the end, and marker 99 (rest) appears
# up to three times. Pooling them would let a single 2 s window straddle the seam
# between two blocks recorded minutes apart. Set to False for the old pooling behaviour.
EPOCH_BY_BLOCK = True

# ── Stage 4: Welch PSD + frequency bands ──────────────────────────────────────
# Welch segment length in samples. 256 @ 256 Hz = 1 s segments -> 1 Hz resolution,
# and 2 averaged segments per 2 s epoch (with 50 % overlap), which trades a little
# frequency detail for a much smoother, lower-variance spectrum estimate.
WELCH_NPERSEG = 256

# Band edges in Hz. Power is the area under the PSD curve between lo and hi.
# One feature is produced per (channel x band) pair -> 4 x 5 = 20 features.
BANDS = {
    "delta": (0.5,  4),   # deep sleep / very slow activity
    "theta": (4,    8),   # drowsiness, inward focus, memory encoding
    "alpha": (8,   13),   # relaxed wakefulness; strongest with eyes closed
    "beta":  (13,  30),   # active, externally directed attention
    "gamma": (30,  44),   # concentration, feature binding (upper edge kept below
                          # LINE_FREQ so the mains notch does not eat into the band)
}

# ── Stage 6: re-referencing ───────────────────────────────────────────────────
# Subtract the mean of these channels from every channel, removing whatever the head
# has in common and leaving what differs between electrodes.
#   ["TP9", "TP10"] — linked mastoid, as used in the previous project and written into
#                     the protocol documents. WARNING: because both reference channels
#                     are themselves in CH_COLS, this makes TP9 and TP10 exact negatives
#                     of each other, so they yield identical band-power features.
#   CH_COLS         — average reference: no pair collapses, all 20 features stay distinct.
#   []              — no re-referencing; keep the Muse's hardware reference (Fpz).
REF_CHANNELS = ["TP9", "TP10"]

# ── Stage 5: metadata ─────────────────────────────────────────────────────────
# Columns copied from Respondents_profiles.csv onto every epoch row.
META_COLS = ["Age", "Gender"]

# ── Marker names (labels only — never used in any computation) ────────────────
# The same code means different things in different protocols: marker 10 is
# "watch sport" in Case 1, "social media" in Case 2, "game" in Case 3 and
# "breath counting" in Case 4. These names are taken from the protocol documents
# in protocols/ and are used to label the quality-control plots.
MARKER_NAMES = {
    1: {1: "eyes open", 2: "eyes closed", 10: "watch sport",
        20: "code solo", 30: "code with AI", 99: "rest"},
    2: {1: "eyes open", 2: "eyes closed", 10: "social media"},
    3: {1: "eyes open", 2: "eyes closed", 10: "game", 20: "shopping",
        30: "market analysis", 40: "IQ test", 99: "rest"},
    4: {1: "eyes open", 2: "eyes closed", 10: "low load (breathing)"},
    5: {1: "eyes open", 2: "eyes closed", 11: "short instructional",
        12: "long instructional", 21: "short entertainment",
        22: "long entertainment", 99: "rest"},
}

# ── Quality-control plots: raw signal vs cleaned signal, over time ────────────
PLOT_BEFORE_AFTER = True    # set False to skip the plotting step entirely

# Which recordings to plot. Plotting all 34 is slow and produces a wall of figures,
# so the default shows one example per case.
#   "first_per_case" — the first recording of each case (a quick overall check)
#   "all"            — every recording (thorough, slow)
#   an integer n     — the first n recordings in the catalogue
#   a list of dicts  — hand-picked, e.g.
#                      [{"subject": "sub-P001", "session": "ses-S001", "case": 2}]
PLOT_WHICH = "first_per_case"

# Time span to draw, in seconds from the start of the recording.
#   None    — the whole recording: every protocol block is visible on the time axis
#   (0, 60) — a zoomed slice at full resolution, for inspecting individual artefacts
PLOT_WINDOW_S = None

# Drawing every sample of a 20-minute recording means ~300,000 points per channel,
# which is slow and renders as a solid block. Above this many points the series is
# decimated (every Nth sample) *for display only* — the data itself is untouched.
# Short, isolated spikes can be missed when decimation is active, so zoom in with
# PLOT_WINDOW_S when inspecting artefacts closely.
PLOT_MAX_POINTS = 20_000

# Figures are rendered inline in the notebook and are NOT written to disk. They are
# diagnostics: quick to regenerate (the whole notebook runs in a few seconds) and they
# would otherwise pile up as stale PNGs that no longer match the current settings.
# Set PLOT_SAVE = True to also write them to PLOT_DIR.
PLOT_SAVE = False
PLOT_DIR  = Path("reports") / "figures" / "preprocessing"

# ── Quality-control plots: Welch periodograms and the feature vectors ─────────
# These show the frequency-domain side — how the Welch transform turns a cleaned
# epoch into the 20 numbers that end up in the CSV. Same PLOT_WHICH selection.
PLOT_PERIODOGRAM = True

# Which figures to draw:
#   "W" — how Welch's method works, walked through step by step on one real epoch
#         (drawn once, for the first selected recording — it explains the method,
#          so there is no point repeating it for every file)
#   "A" — full-recording periodogram, before vs after preprocessing
#         (shows what the bandpass and the notch actually did to the spectrum)
#   "B" — epoch-level periodogram: every accepted epoch's PSD, averaged per block
#         (this is the Welch transform the features are read off)
#   "C" — the resulting feature vectors
PSD_FIGURES = ("W", "A", "B", "C")

# Channel used for the figure W walk-through. A frontal electrode usually has the
# clearest structure; any name from CH_COLS works.
PSD_DEMO_CHANNEL = "AF7"

# Segment length for the full-recording PSD in figure A, in samples.
# 4 s gives 0.25 Hz resolution — finer than the per-epoch estimate, which is what
# you want when the question is "did the filters do their job".
PSD_RECORDING_NPERSEG = int(SFREQ * 4)

# Upper limit of the frequency axis. A little above LINE_FREQ so the mains notch
# is visible in figure A.
PSD_FMAX = 60

# Colour per frequency band, reused across every figure so a band always looks
# the same. Labels only — never used in any computation.
BAND_COLORS = {
    "delta": "#4C72B0",   # blue
    "theta": "#55A868",   # green
    "alpha": "#DD8452",   # orange
    "beta":  "#C44E52",   # red
    "gamma": "#8172B3",   # purple
}

# ── Performance ───────────────────────────────────────────────────────────────
# Recordings are processed in parallel across CPU cores. Each recording is fully
# independent — load, clean and featurise touch no shared state — so this is a
# straight division of work with no coordination between workers.
#   -1 = use every core        (fastest)
#    n = use n cores
#    1 = no parallelism at all (sequential; use this when debugging, because
#        errors then surface with a normal traceback instead of being relayed
#        back from a worker process)
N_JOBS = -1

# ── Output ────────────────────────────────────────────────────────────────────
OUT_PATTERN = "Case{case}_epochs.csv"   # {case} -> 1, 2, 3, ...
FLOAT_FMT   = "%.6g"                    # CSV float precision (keeps files small)

print("[SETTINGS] loaded.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [SETUP]  —  imports, project root, small compatibility helpers
# ══════════════════════════════════════════════════════════════════════════════
import os
import re
import sys
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import pyxdf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch
from joblib import Parallel, delayed, cpu_count

# ── Locate the project root ───────────────────────────────────────────────────
# The notebook lives in "02. Code/", but the data lives one level up. Walk upwards
# from the current working directory until we find the folder that contains
# "01. Data", so the notebook works whether Jupyter was started in the project
# root, in "02. Code/", or anywhere in between.
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "01. Data").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find a folder containing '01. Data' at or above {start}.\n"
        "Start Jupyter from the project root or from '02. Code/'."
    )

PROJECT_ROOT = find_project_root(Path(os.getcwd()).resolve())

# Turn the short relative paths from [SETTINGS] into absolute ones.
RAW_PATH      = PROJECT_ROOT / RAW_DIR
OUT_PATH      = PROJECT_ROOT / OUT_DIR
PROFILES_PATH = PROJECT_ROOT / PROFILES_CSV
OUT_PATH.mkdir(parents=True, exist_ok=True)   # create the output folder if missing

# ── Derived constants ─────────────────────────────────────────────────────────
EPOCH_N    = int(EPOCH_S * SFREQ)                       # epoch length in samples (512)
FEAT_COLS  = [f"{ch}_{band}" for ch in CH_COLS for band in BANDS]   # the 20 feature names
LABEL_COLS = ["subject", "session", "case", "marker", "block", "epoch_idx"]

# ── numpy compatibility ───────────────────────────────────────────────────────
# np.trapz was renamed to np.trapezoid in NumPy 2.0. Bind whichever exists so the
# notebook runs on both.
TRAPZ = getattr(np, "trapezoid", None) or np.trapz

print(f"[SETUP] Project root : {PROJECT_ROOT}")
print(f"[SETUP] Raw data     : {RAW_PATH}")
print(f"[SETUP] Output       : {OUT_PATH}")
print(f"[SETUP] Epoch length : {EPOCH_S} s = {EPOCH_N} samples @ {SFREQ:.0f} Hz")
print(f"[SETUP] Features     : {len(FEAT_COLS)} "
      f"({len(CH_COLS)} channels x {len(BANDS)} bands)")
print(f"[SETUP] CPU cores    : {cpu_count()} available, "
      f"N_JOBS = {N_JOBS} ({'all cores' if N_JOBS == -1 else 'sequential' if N_JOBS == 1 else str(N_JOBS) + ' cores'})")
print(f"[SETUP] numpy {np.__version__} | pandas {pd.__version__} | pyxdf {pyxdf.__version__}")

## Step 1 — Discover the recordings

### What the code does

`discover_recordings()` walks the raw-data tree and returns a **catalogue**: one row per
`.xdf` file, recording who it came from, which session, and which case. Nothing is read
from inside the files yet — this step only decides *what* will be processed, so a mistake
here is cheap to spot and fix.

The folder layout is BIDS-style, and every piece of identity is encoded in the path:

```
01. Data/00. Raw/ sub-P001 / ses-S001 / eeg / sub-P001_ses-S001_task-Case2_run-001_eeg.xdf
                  ^^^^^^^^   ^^^^^^^^                                   ^^^^^     ^^^
                  subject    session                                    case      run
```

The scan is three nested loops — every `sub-*` folder, every `ses-*` inside it, then every
case in `CASES` — matching files against `FILE_GLOB` (`*task-Case2_run-*_eeg.xdf` and so on).
Sessions with no `eeg/` subfolder are skipped silently, since an empty session folder is a
normal artefact of collection rather than an error.

Every loop uses `sorted()`. Filesystem iteration order is not guaranteed, and without
sorting the catalogue — and therefore the row order of the output CSVs — could differ
between machines or between runs on the same machine.

### Why identity comes from the folders

**Subject and session are read from the folder names, not from the filename.** The two
normally agree, but when they disagree one of them has to win, and the folder is the better
witness: it reflects where the recording was actually filed, whereas the filename is typed
during collection and can carry a typo.

The code cross-checks anyway and prints a warning naming both values, because a mismatch is
usually a sign that something went wrong at collection time and is worth reconciling against
the lab notes. It is a warning rather than an error — a misnamed file is still perfectly
good data, and stopping the batch over it would help nobody.

> **This fires on your data.** `sub-P008/ses-S002/eeg/` contains a file *named*
> `sub-P008_ses-S001_...`. It is catalogued as session S002, from its folder.

### Output

A `catalogue` DataFrame (`subject`, `session`, `case`, `run`, `path`, `name`), followed by
a count per case and a subject × case matrix of how many sessions each person contributed.
That matrix is the quickest way to see which analyses are actually possible: Case 2's
session model needs subjects with two sessions, and the matrix shows at a glance who has them.

If the catalogue comes back empty the code raises immediately rather than letting later
steps fail confusingly — the cause is nearly always a wrong `RAW_DIR` or `FILE_GLOB`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 1]  Discover every raw recording
# ══════════════════════════════════════════════════════════════════════════════

def discover_recordings(raw_path: Path, cases: list) -> pd.DataFrame:
    """
    Scan the raw-data tree and return one row per .xdf file found.

    Expected layout:  raw_path / sub-PXXX / ses-SXXX / eeg / *task-CaseN_run-*_eeg.xdf

    Parameters
    ----------
    raw_path : Path        root of the raw data tree
    cases    : list[int]   case numbers to look for, e.g. [1, 2, 3, 4, 5]

    Returns
    -------
    pd.DataFrame with columns: subject, session, case, run, path, name
    """
    rows = []

    # sorted() everywhere keeps the catalogue deterministic between runs.
    for sub_dir in sorted(raw_path.glob("sub-*")):
        if not sub_dir.is_dir():
            continue

        for ses_dir in sorted(sub_dir.glob("ses-*")):
            eeg_dir = ses_dir / "eeg"
            if not eeg_dir.is_dir():
                continue   # session folder exists but holds no eeg/ subfolder

            for case in cases:
                pattern = FILE_GLOB.format(case=f"Case{case}")

                for xdf_path in sorted(eeg_dir.glob(pattern)):
                    # ── cross-check the filename against the folder names ──────
                    # The folder wins; a mismatch only produces a warning.
                    name_sub = re.search(r"(sub-P\d+)", xdf_path.name)
                    name_ses = re.search(r"(ses-S\d+)", xdf_path.name)
                    if name_sub and name_sub.group(1) != sub_dir.name:
                        print(f"  ! WARNING  subject mismatch: file says "
                              f"'{name_sub.group(1)}' but sits in folder "
                              f"'{sub_dir.name}'  ->  using the folder.")
                    if name_ses and name_ses.group(1) != ses_dir.name:
                        print(f"  ! WARNING  session mismatch: file says "
                              f"'{name_ses.group(1)}' but sits in folder "
                              f"'{ses_dir.name}'  ->  using the folder.")

                    # run number, e.g. "run-001" -> 1  (defaults to 1 if absent)
                    run_match = re.search(r"run-(\d+)", xdf_path.name)
                    run = int(run_match.group(1)) if run_match else 1

                    rows.append({
                        "subject": sub_dir.name,
                        "session": ses_dir.name,
                        "case":    case,
                        "run":     run,
                        "path":    xdf_path,
                        "name":    xdf_path.name,
                    })

    return pd.DataFrame(rows)


print("[STEP 1] Scanning for recordings...\n")
catalogue = discover_recordings(RAW_PATH, CASES)

if catalogue.empty:
    raise RuntimeError(
        f"No .xdf files found under {RAW_PATH}.\n"
        f"Check RAW_DIR, CASES and FILE_GLOB in the [SETTINGS] cell."
    )

# ── Overview: how many recordings per case, and per subject ───────────────────
print(f"\n[STEP 1] Found {len(catalogue)} recording(s) "
      f"from {catalogue['subject'].nunique()} subject(s).\n")

print("  Recordings per case:")
for case, n in catalogue["case"].value_counts().sort_index().items():
    n_subs = catalogue.loc[catalogue["case"] == case, "subject"].nunique()
    print(f"    Case {case} : {n:>3} recording(s) from {n_subs} subject(s)")

print("\n  Subject x case matrix (number of sessions):")
display(
    catalogue.pivot_table(index="subject", columns="case",
                          values="session", aggfunc="nunique", fill_value=0)
)

## Step 2 — Load one XDF file

### The problem this step solves

An XDF file is not a table. It holds several independent **streams** that LabRecorder
recorded in parallel, each with its own sample rate and its own timestamps:

| Stream type | Content | Rate |
|---|---|---|
| `EEG` | The 4 electrode channels, plus an unused `Right AUX` | 256 Hz, regular |
| `Markers` | Integer event codes pushed at each block transition | irregular — one per block |
| (others) | PPG, accelerometer, gyroscope | various |

The EEG stream is a continuous river of numbers with no notion of what the subject was
doing; the marker stream knows what was happening but contains only a handful of events.
**Neither is useful alone.** This step joins them into one table where every EEG sample
carries the label of the block it belongs to.

### How the code does it

1. **`pyxdf.load_xdf()`** reads every stream. `find_stream()` then picks the EEG and Markers
   streams by their declared `type` field rather than by position, since stream order in the
   file is not guaranteed.
2. **`channel_labels()`** reads the electrode names out of the stream's XML header, falling
   back to `ch0, ch1, …` if the header carries none. The frame is then narrowed to the four
   channels in `CH_COLS`, which drops `Right AUX` (nothing is plugged into it). Requesting a
   channel that does not exist raises with the list of what *is* available.
3. **Both streams are rebased onto a common clock.** LSL timestamps are seconds from an
   arbitrary origin — typically a large number like 92847.331. The code finds `t0`, the
   earliest timestamp across *all* streams in the file, and subtracts it from both the EEG
   and the marker timestamps. This does two jobs: it makes `time_s` readable (0 = start of
   recording), and, because the *same* `t0` is removed from both, it keeps them on one axis
   so the join below is correct.
4. **`pd.merge_asof(..., direction="backward")`** does the actual alignment. An ordinary join
   would be useless here — markers and EEG samples almost never share an exact timestamp.
   An "as-of" join instead matches each EEG sample to the *most recent marker at or before*
   it, which is exactly the semantics wanted: a marker announces a block, and everything
   after it belongs to that block until the next marker arrives.

```
markers:      10 ──────────────► 99 ──────────────► 20 ─────►
EEG samples:  ● ● ● ● ● ● ● ● ●  ● ● ● ● ● ● ● ● ●  ● ● ● ●
labelled as: 10 10 10 10 ...     99 99 99 ...       20 20 ...
```

   `merge_asof` requires both sides sorted on the join key, so both frames are sorted first.

5. **The lead-in is trimmed.** The headband streams as soon as it connects, so the samples
   before the first marker were recorded while the experimenter was still setting up. They
   come out of the join with `marker = NaN`; `notna().idxmax()` finds the first labelled row
   and everything before it is dropped.

### Output

One `DataFrame` per recording: `time_s`, the four channels, and `marker` — the shape every
later step expects. Nothing is filtered or cleaned yet; this is the raw signal, merely
labelled and given a readable clock.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 2]  XDF loading helpers
# ══════════════════════════════════════════════════════════════════════════════

def find_stream(streams: list, stream_type: str) -> dict:
    """Return the first stream whose declared type matches (case-insensitively)."""
    for stream in streams:
        if stream["info"]["type"][0].lower() == stream_type.lower():
            return stream
    raise ValueError(
        f"No stream of type '{stream_type}' in this file. "
        f"Available: {[s['info']['type'][0] for s in streams]}"
    )


def channel_labels(stream: dict) -> list:
    """
    Read the channel names out of a stream's XML header.
    Falls back to ch0, ch1, ... when the header carries no labels.
    """
    desc = stream["info"].get("desc", [{}])
    if not desc or not desc[0]:
        n_ch = int(stream["info"]["channel_count"][0])
        return [f"ch{i}" for i in range(n_ch)]
    channels = desc[0].get("channels", [{}])[0].get("channel", [])
    return [ch["label"][0] for ch in channels]


def load_xdf(xdf_path: Path, ch_cols: list) -> pd.DataFrame:
    """
    Load one XDF recording into a single time-aligned DataFrame.

    Returns
    -------
    pd.DataFrame with columns: time_s, <ch_cols...>, marker
        time_s : float, seconds since the start of the recording
        marker : int,   the event code in force at that sample
    """
    streams, _header = pyxdf.load_xdf(str(xdf_path))

    eeg_stream    = find_stream(streams, "EEG")
    marker_stream = find_stream(streams, "Markers")

    # ── EEG stream -> DataFrame, keeping only the requested channels ───────────
    labels = channel_labels(eeg_stream)
    eeg_df = pd.DataFrame(eeg_stream["time_series"], columns=labels)
    eeg_df.insert(0, "time_s", eeg_stream["time_stamps"])

    missing = [c for c in ch_cols if c not in eeg_df.columns]
    if missing:
        raise ValueError(
            f"Channels {missing} are not in this recording. "
            f"Available: {eeg_df.columns.tolist()}"
        )
    eeg_df = eeg_df[["time_s"] + ch_cols].copy()

    # ── Rebase both streams to "seconds since recording start" ────────────────
    # t0 is the earliest timestamp across *all* streams in the file. Subtracting the
    # same t0 from EEG and marker timestamps keeps them on a common axis, which is
    # what makes the merge below correct.
    t0 = min(s["time_stamps"][0] for s in streams if len(s["time_stamps"]) > 0)
    eeg_df["time_s"] = eeg_df["time_s"] - t0

    # ── Markers stream -> DataFrame ───────────────────────────────────────────
    # Each marker sample is a 1-element list holding the code as a string.
    marker_vals = [int(m[0]) if m else np.nan for m in marker_stream["time_series"]]
    markers_df = pd.DataFrame({
        "time_s": marker_stream["time_stamps"] - t0,
        "marker": marker_vals,
    })

    # ── Time-align: give every EEG sample the last marker seen before it ───────
    # merge_asof needs both sides sorted on the join key.
    eeg_df     = eeg_df.sort_values("time_s").reset_index(drop=True)
    markers_df = markers_df.sort_values("time_s").reset_index(drop=True)
    merged = pd.merge_asof(eeg_df, markers_df, on="time_s", direction="backward")

    # ── Drop the lead-in recorded before the first marker ─────────────────────
    # Those samples have marker = NaN: the headband was already streaming but the
    # protocol had not started yet.
    first_valid = merged["marker"].notna().idxmax()
    merged = merged.loc[first_valid:].reset_index(drop=True)

    return merged


print("[STEP 2] XDF loading helpers defined.")

## Step 3 — Preprocess (6 stages)

Raw EEG from a consumer headband is not usable as recorded. It carries mains hum, drifting
electrode offsets, dropped samples, and voltage swings from jaw and eye movement that are
an order of magnitude larger than the brain activity underneath. This step is the cleaning
pipeline pre-registered in every protocol document, applied in this exact order.

### The six stages

| # | Stage | What it removes | Why it is done this way |
|---|---|---|---|
| 1 | Muse sentinels → `NaN` | Fake samples | When the Bluetooth link drops a packet the firmware writes a fixed magic number (`-1000.0` or `999.511719`) rather than leaving a gap. These are not measurements. Left in place they would be read as real ±1000 µV excursions and would wreck every stage below. |
| 2 | KNN interpolation | The resulting gaps | `sosfiltfilt` and `filtfilt` cannot process `NaN`: a single missing sample propagates through the filter and destroys the whole channel. The gaps are filled from their nearest valid neighbours, weighted by inverse distance so an adjacent sample counts far more than a distant one. |
| 3 | Band-pass 1–50 Hz | Drift below, noise above | Below 1 Hz sits the electrode DC offset and slow drift from sweat and movement. Above 50 Hz there is nothing this study uses. Both are removed by a Butterworth filter. |
| 4 | Notch at 50 Hz | Mains hum | Room wiring radiates at 50 Hz (Europe; 60 Hz in North America) and the headband picks it up. In your recordings this peak reaches 10⁶ µV²/Hz — larger than the brain signal by orders of magnitude. A narrow notch removes it while leaving neighbouring frequencies intact. |
| 5 | \|x\| > 150 µV → `NaN` → KNN | Surviving artefacts | Jaw clenches, blinks and cable tugs produce excursions far larger than cortical signal. They are blanked and interpolated rather than deleted, so the time axis stays continuous and every later step can assume a uniform 256 Hz. |
| 6 | Linked-mastoid re-reference | Signal common to the whole head | Every electrode measures its own site *plus* whatever the shared reference picks up. Subtracting `mean(TP9, TP10)` removes what is common to the head, leaving what differs between electrodes. |

### Why the order matters

**Filtering comes before the amplitude threshold.** A raw signal riding on a −200 µV DC
offset would exceed a ±150 µV test at literally every sample; the threshold is only
meaningful once the offset and drift have been filtered out. This is why stages 3–4 sit
between the two interpolation passes.

**KNN interpolation runs twice**, for two different reasons: first to fill the dropped
packets so the filters can run at all, then again to fill the artefacts that only became
identifiable after filtering.

### Two implementation details

**Zero-phase filtering.** `sosfiltfilt` runs the filter forwards and then backwards over the
signal. Any ordinary filter delays the signal, and delays it by different amounts at
different frequencies — which would shift alpha in time relative to beta and corrupt any
comparison between bands. Running it in both directions cancels the delay exactly. The cost
is that the effective filter order doubles, which is why `BANDPASS_ORDER = 4` behaves like
an 8th-order filter. The second-order-sections (`sos`) form is used because it stays
numerically stable where the older transfer-function form degrades.

**The reference is computed once, before the loop.** `reference = clean[REF_CHANNELS].mean(axis=1)`
is evaluated and stored *before* any channel is modified. Updating channels in place while
computing the reference from them would mean later channels get referenced to already-shifted
values — a subtle bug that would silently corrupt every downstream feature.

> ### ⚠ A consequence of stage 6 worth knowing
> With `REF_CHANNELS = ["TP9", "TP10"]`, both reference channels are themselves in `CH_COLS`,
> and the arithmetic works out as:
>
> ```
> reference = (TP9 + TP10) / 2
> TP9'  = TP9  - reference =  (TP9 - TP10) / 2
> TP10' = TP10 - reference = -(TP9 - TP10) / 2  =  -TP9'
> ```
>
> **The two channels become exact negatives of each other.** Band power squares the signal,
> so the sign cancels and TP9 and TP10 yield the same numbers — equal to within
> floating-point rounding (~1e-15 relative, and byte-identical once written to CSV at six
> significant figures). Only **15 of the 20 features carry independent information**, and the
> separate temporal-lobe signal that Case 4's hypothesis refers to is gone: what remains is
> a single bipolar TP9−TP10 difference.
>
> This is inherited from the previous project (`pipeline/preprocessing/preprocess.py`) and is
> written into all five protocol documents, so it is left as-is here rather than changed
> silently. To depart from the protocol, set `REF_CHANNELS = CH_COLS` for an average
> reference or `REF_CHANNELS = []` to keep the Muse's hardware reference — both verified to
> leave all 20 features distinct.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 3]  Signal-cleaning helpers
# ══════════════════════════════════════════════════════════════════════════════

def knn_interpolate(arr: np.ndarray, k: int = 5) -> np.ndarray:
    """
    Fill NaN entries of a 1-D array from their k nearest valid neighbours.

    "Distance" is the gap in sample index, and neighbours are weighted by 1/distance,
    so a sample right next to the gap counts far more than one 50 samples away.

    Implementation note
    -------------------
    The obvious version — for each NaN, measure the distance to every valid sample and
    keep the k smallest — is O(n_NaN x n_valid) and takes minutes on a 10-minute
    recording. It is also unnecessary: because the valid indices are sorted, the
    distance |valid_idx[j] - i| is V-shaped in j, so the k nearest neighbours are always
    within k positions either side of where i would be inserted into valid_idx. Finding
    that insertion point with a binary search (np.searchsorted) narrows the search from
    "every valid sample" to "2k candidates", giving identical results in O(n_NaN x k).

    Parameters
    ----------
    arr : 1-D float array, may contain NaN
    k   : how many valid neighbours to average

    Returns
    -------
    A copy of arr with the NaNs replaced.
    """
    arr       = arr.copy()
    nan_idx   = np.flatnonzero(np.isnan(arr))
    valid_idx = np.flatnonzero(~np.isnan(arr))

    # Nothing to fill, or nothing to fill *from* — return unchanged.
    if nan_idx.size == 0 or valid_idx.size == 0:
        return arr

    k_use = min(k, valid_idx.size)      # clamp: fewer valid samples than k

    # Where each NaN would slot into the sorted list of valid indices.
    pos = np.searchsorted(valid_idx, nan_idx)

    # Candidate neighbours: k positions either side of that slot. (n_NaN, 2k)
    offsets   = np.arange(-k_use, k_use)
    cand_pos  = pos[:, None] + offsets[None, :]

    # Near the start/end of the recording some candidates fall outside the array.
    # Clip them to stay in bounds for indexing, then give them an infinite distance
    # so their weight becomes zero and they contribute nothing to the average.
    in_bounds = (cand_pos >= 0) & (cand_pos < valid_idx.size)
    cand_pos  = np.clip(cand_pos, 0, valid_idx.size - 1)
    cand_idx  = valid_idx[cand_pos]                              # sample indices

    dists = np.abs(cand_idx - nan_idx[:, None]).astype(float)
    dists[~in_bounds] = np.inf

    # Keep the k nearest of the 2k candidates.
    # Ties are common and must be broken explicitly: an isolated NaN sees distances
    # 1, 1, 2, 2, 3, 3, ... so with an odd k the k-th neighbour could be taken from
    # either side. Candidates are laid out in ascending sample order, so a *stable*
    # sort resolves every tie toward the earlier sample — deterministic, and identical
    # from run to run and machine to machine.
    chosen = np.argsort(dists, axis=1, kind="stable")[:, :k_use]
    rows   = np.arange(nan_idx.size)[:, None]

    weights = 1.0 / (dists[rows, chosen] + 1e-8)   # +1e-8 guards against divide-by-zero
    values  = arr[cand_idx[rows, chosen]]          # only ever reads non-NaN samples

    arr[nan_idx] = (weights * values).sum(axis=1) / weights.sum(axis=1)

    return arr


def bandpass(arr: np.ndarray, lo: float, hi: float, fs: float, order: int = 4) -> np.ndarray:
    """
    Zero-phase Butterworth band-pass, applied down each column of arr (n_samples, n_ch).

    Second-order-sections (`sos`) form is numerically stable at high orders, and
    `sosfiltfilt` runs the filter forwards then backwards so the net phase shift is zero
    — essential when the features depend on relative timing across bands.
    """
    sos = butter(order, [lo, hi], btype="band", fs=fs, output="sos")
    return sosfiltfilt(sos, arr, axis=0)


def notch(arr: np.ndarray, freq: float, fs: float, Q: float = 30) -> np.ndarray:
    """
    Zero-phase IIR notch at `freq` Hz, applied down each column of arr.

    `iirnotch` returns (b, a) transfer-function coefficients rather than sos, so this
    uses `filtfilt` (the transfer-function equivalent of sosfiltfilt).
    """
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, arr, axis=0)


def preprocess(eeg_df: pd.DataFrame) -> pd.DataFrame:
    """
    Run all 6 cleaning stages on the output of load_xdf().

    Returns a DataFrame of the same shape, with the channel columns replaced by
    their cleaned versions. time_s and marker pass through untouched.
    """
    clean = eeg_df.copy()

    # ── Stage 1: Muse sentinel values -> NaN ──────────────────────────────────
    for col in CH_COLS:
        clean[col] = clean[col].replace(SENTINELS, np.nan)

    # ── Stage 2: fill those gaps so the filters see a complete signal ─────────
    for col in CH_COLS:
        clean[col] = knn_interpolate(clean[col].values.astype(float), k=KNN_K)

    # ── Stage 3 + 4: band-pass, then notch out the mains frequency ────────────
    arr = clean[CH_COLS].values.astype(float)      # (n_samples, n_channels)
    arr = bandpass(arr, BANDPASS_LO, BANDPASS_HI, SFREQ, order=BANDPASS_ORDER)
    arr = notch(arr, LINE_FREQ, SFREQ, Q=NOTCH_Q)
    clean[CH_COLS] = arr

    # ── Stage 5: blank out remaining large excursions and re-interpolate ──────
    artefact_mask  = np.abs(clean[CH_COLS]) > THRESHOLD
    clean[CH_COLS] = clean[CH_COLS].where(~artefact_mask)    # True -> NaN
    for col in CH_COLS:
        clean[col] = knn_interpolate(clean[col].values.astype(float), k=KNN_K)

    # ── Stage 6: re-reference ─────────────────────────────────────────────────
    # An empty REF_CHANNELS means "keep the hardware reference" — skip this stage.
    if REF_CHANNELS:
        # Computed once, before the loop, so every channel is referenced to the same
        # signal. Updating in place would reference the later channels to values that
        # have already been shifted, which silently corrupts every downstream feature.
        reference = clean[REF_CHANNELS].mean(axis=1)
        for col in CH_COLS:
            clean[col] = clean[col] - reference

    return clean


print("[STEP 3] Preprocessing helpers defined.")

## Step 4 — Epoching and Welch band-power features

### Why epoch at all

A model needs a table: fixed-size rows, each with a label. A recording is the opposite —
one continuous stream of unequal length per subject. Epoching converts the second into the
first by chopping the signal into equal windows, each of which becomes one row and inherits
the label of the block it came from. It also multiplies the sample size: a 6-minute
recording yields ~180 rows rather than 1.

The window length is a trade-off. Too short and the spectrum estimate is unreliable and
cannot resolve low frequencies (a 2 s window is the minimum for a meaningful 0.5 Hz delta
measurement). Too long and you get too few rows, and any brief artefact contaminates a large
window. `EPOCH_S = 2.0` follows the previous project.

### How the code cuts them

`extract_epochs()` groups the cleaned frame by **contiguous marker block**, then slices each
block into non-overlapping windows of `EPOCH_N = 512` samples, discarding the remainder at
the end of each block (a partial window cannot produce a comparable spectrum).

Blocks are identified with `marker.ne(marker.shift()).cumsum()` — a running count that
increments every time the marker value changes, giving 1, 1, 1, 2, 2, 3… This matters
because **several protocols reuse a marker code**: marker 1 (eyes open) runs at the start
*and* again at the end, and marker 99 (rest) appears up to three times. Grouping by marker
*value* alone would pool blocks recorded minutes apart and let a single window straddle the
seam between them, producing an epoch that is half one condition and half another. Grouping
by contiguous block prevents that, and the resulting `block` number is kept as a column so
later analyses can tell the opening baseline from the closing one.

Setting `EPOCH_BY_BLOCK = False` restores the previous project's pooling behaviour.

### Artefact rejection

A window is discarded if **any** channel's peak-to-peak amplitude exceeds `PTP_THRESH`
(420 µV) — `np.ptp(window, axis=0).max()` takes the worst channel. Preprocessing stage 5
already repaired isolated spikes; this catches windows where the signal was disturbed for
long enough that interpolation cannot be trusted, and it is cheaper to drop such a window
than to feed a model a spectrum computed from repaired noise.

The cost is visible in the `kept_%` column of the Step 10 tables: on clean recordings 95–99 %
of windows survive, but a badly disturbed recording can lose almost everything.

### From window to features

Each surviving window is turned into a spectrum with **Welch's method** (`welch_psd()`), then
the power in each frequency band is measured as the **area under the PSD curve** between the
band edges, by trapezoidal integration, in µV²/Hz:

```
       PSD
        |        ___
        |   ____/   \____            band power = the shaded area
        |__/  ###########  \____                  between lo and hi
        |     ###########
        +-----|---------|------------ Hz
             lo         hi
```

Doing this for every (channel × band) pair gives the **20 numbers that describe one epoch** —
one row of the output CSV. Why band power rather than the raw spectrum: it compresses ~50
frequency bins per channel down to 5 numbers, and the bands correspond to rhythms with known
functional meaning, so a model trained on them is interpretable.

**Step 8 visualises this whole process on a real epoch** — Welch's method is explained in
full there rather than repeated here.

### Output

`extract_features()` returns one row per accepted epoch: `marker`, `block`, `epoch_idx` and
the 20 feature columns. If every epoch was rejected it returns an *empty frame with the right
columns*, not a bare `DataFrame()` — so concatenating results across recordings cannot fail
on a mismatched schema.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 4]  Epoching + Welch band-power features
# ══════════════════════════════════════════════════════════════════════════════

def extract_epochs(clean_df: pd.DataFrame) -> list:
    """
    Cut the cleaned recording into fixed-length, non-overlapping epochs.

    Returns
    -------
    list of dicts, one per accepted epoch:
        {"marker": int, "block": int, "epoch_idx": int, "data": (EPOCH_N, n_ch) array}

    `block` numbers the contiguous runs of a constant marker value, in time order
    (1, 2, 3, ...). It lets a later analysis tell the opening eyes-open block apart
    from the closing one even though both carry marker 1.
    """
    df = clean_df.copy()

    if EPOCH_BY_BLOCK:
        # A new block starts wherever the marker value differs from the previous row.
        # cumsum over that boolean gives 1, 1, 1, 2, 2, 3, ... — a contiguous block id.
        marker_series  = df["marker"]
        df["block"]    = marker_series.ne(marker_series.shift()).cumsum()
        group_keys     = ["block", "marker"]
    else:
        # Legacy behaviour: pool every sample sharing a marker value, regardless of
        # when it was recorded. Kept for comparison with the previous project.
        df["block"] = 0
        group_keys  = ["marker"]

    epochs = []

    for keys, group in df.groupby(group_keys, sort=True):
        # Unpack the group key(s) into block / marker, whichever grouping was used.
        if EPOCH_BY_BLOCK:
            block, marker = keys
        else:
            block, marker = 0, keys if np.isscalar(keys) else keys[0]

        signal    = group[CH_COLS].values          # (n_samples_in_block, n_channels)
        n_windows = len(signal) // EPOCH_N         # only complete windows; remainder dropped

        for i in range(n_windows):
            window = signal[i * EPOCH_N : (i + 1) * EPOCH_N]

            # Peak-to-peak rejection: .max() takes the worst channel in the window.
            if PTP_THRESH is not None and np.ptp(window, axis=0).max() > PTP_THRESH:
                continue

            epochs.append({
                "marker":    int(marker),
                "block":     int(block),
                "epoch_idx": i,          # position of this window inside its block
                "data":      window,
            })

    return epochs


def welch_psd(data: np.ndarray, nperseg: int):
    """
    Welch power spectral density estimate.

    The single place Welch is called, so the periodograms drawn in Step 8 are by
    construction the very same transform the features are computed from — they
    cannot drift apart.

    Parameters
    ----------
    data    : (n_samples, n_channels) array
    nperseg : segment length in samples; capped at the data length so short
              inputs cannot error out

    Returns
    -------
    freqs : (n_freqs,)            frequency axis in Hz
    psd   : (n_channels, n_freqs) power spectral density in uV^2/Hz
    """
    nperseg = int(min(nperseg, data.shape[0]))
    # welch expects (n_channels, n_samples), hence the transpose.
    # noverlap = nperseg // 2 (50 % overlap) and a Hann window are scipy's defaults,
    # written out here so the settings are visible rather than implied.
    freqs, psd = welch(data.T, fs=SFREQ, nperseg=nperseg, noverlap=nperseg // 2,
                       window="hann", scaling="density")
    return freqs, psd


def band_power(data: np.ndarray) -> dict:
    """
    Welch PSD of one epoch, integrated over each frequency band.

    Parameters
    ----------
    data : (EPOCH_N, n_channels) array — one epoch of cleaned EEG

    Returns
    -------
    dict {"TP9_delta": float, "TP9_theta": float, ...} — one entry per channel x band
    """
    freqs, psd = welch_psd(data, WELCH_NPERSEG)   # psd: (n_channels, n_freqs)

    features = {}
    for band_name, (lo, hi) in BANDS.items():
        # Frequency bins that fall inside this band.
        idx = np.where((freqs >= lo) & (freqs <= hi))[0]
        for ch_i, ch_name in enumerate(CH_COLS):
            # Trapezoidal integration = area under the PSD curve across the band.
            features[f"{ch_name}_{band_name}"] = float(TRAPZ(psd[ch_i, idx], freqs[idx]))

    return features


def extract_features(clean_df: pd.DataFrame) -> pd.DataFrame:
    """
    Full epoch -> feature-table step: cut into epochs, then describe each one.

    Returns
    -------
    pd.DataFrame, one row per accepted epoch, with columns
    marker, block, epoch_idx and the 20 band-power features.
    Empty DataFrame if every epoch was rejected.
    """
    epochs = extract_epochs(clean_df)

    if not epochs:
        return pd.DataFrame(columns=["marker", "block", "epoch_idx"] + FEAT_COLS)

    rows = []
    for ep in epochs:
        row = band_power(ep["data"])
        row["marker"]    = ep["marker"]
        row["block"]     = ep["block"]
        row["epoch_idx"] = ep["epoch_idx"]
        rows.append(row)

    return pd.DataFrame(rows)


print("[STEP 4] Feature-extraction helpers defined.")
print(f"          Feature columns ({len(FEAT_COLS)}): {', '.join(FEAT_COLS)}")

## Step 5 — Respondent metadata

### What the code does

`Respondents_profiles.csv` records one row per *recording session*, holding demographics
(age, gender) alongside state variables (hours of sleep, coffees drunk, alcohol in the last
24 h, time of day). `load_person_metadata()` reduces it to a small lookup table — one row per
subject, carrying just the columns named in `META_COLS` — which Step 9 joins onto the epochs.

The identifiers differ between the two sources: the CSV uses `P001` while the folders use
`sub-P001`. The code prefixes `sub-` so the join key matches the rest of the notebook.

### Why the lookup is keyed on the person, not the session

Age and gender are properties of the **person**, not of any one recording, so the code groups
by `Person_code` alone rather than by `(Person_code, Session, Case)`.

That choice is deliberate rather than lazy. Your profiles file contains a handful of rows
that are duplicated or keyed inconsistently — `P006 / S002 / Case 2` and `P008 / S002 / Case 2`
each appear twice, and there is a `P002 / S003` with no corresponding folder on disk. Keying
on the session would mean deciding what to do about every one of those. Keying on the person
sidesteps the problem entirely, because the answer is the same regardless of which duplicate
row you read.

For each subject the code takes the distinct non-null values of each column and uses the
first. If a subject's rows **disagree** — the same person recorded as two different ages —
that is a data-entry error, so it prints a warning naming the subject, the conflicting
values, and which one it used.

### Coverage checks

Two mismatches are reported, because each means something different:

- **Recorded but not in the profiles file** — a warning. Those epochs would get `NaN` for age
  and gender, which would quietly break any analysis grouping on them.
- **In the profiles file but not recorded** — a note, not a warning. This is normal: a
  session can be logged and then not completed.

### Scope

Only **Age and Gender** are carried through. The state variables (sleep, caffeine, alcohol)
stay in `Respondents_profiles.csv`; join them from there if a later analysis wants them as
covariates. Add them to `META_COLS` if you would rather have them on every row — but note
they are genuinely per-session, so the person-level lookup used here would be the wrong tool.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 5]  Load Age + Gender, keyed by subject
# ══════════════════════════════════════════════════════════════════════════════

def load_person_metadata(profiles_path: Path) -> pd.DataFrame:
    """
    Build a per-subject lookup table of the columns listed in META_COLS.

    Returns
    -------
    pd.DataFrame indexed by subject id ("sub-P001"), with one column per META_COLS entry.
    """
    profiles = pd.read_csv(profiles_path)

    # "P001" in the CSV -> "sub-P001" to match the folder names used everywhere else.
    profiles["subject"] = "sub-" + profiles["Person_code"].astype(str).str.strip()

    records = {}
    for subject, group in profiles.groupby("subject"):
        record = {}
        for col in META_COLS:
            # Distinct non-null values recorded for this person across all their sessions.
            values = group[col].dropna().unique()

            if len(values) == 0:
                record[col] = np.nan
            else:
                record[col] = values[0]     # take the first as authoritative
                if len(values) > 1:
                    print(f"  ! WARNING  {subject} has conflicting {col} values "
                          f"{list(values)} — using {values[0]}.")
        records[subject] = record

    return pd.DataFrame.from_dict(records, orient="index").rename_axis("subject")


print("[STEP 5] Loading respondent metadata...\n")
person_meta = load_person_metadata(PROFILES_PATH)

print(f"\n[STEP 5] Metadata for {len(person_meta)} subject(s):")
display(person_meta)

# Flag anyone who has recordings but no profile row — they would get NaN metadata.
recorded  = set(catalogue["subject"])
profiled  = set(person_meta.index)
if recorded - profiled:
    print(f"  ! WARNING  recorded but missing from the profiles file: "
          f"{sorted(recorded - profiled)}")
if profiled - recorded:
    print(f"  (note) in the profiles file but with no recordings on disk: "
          f"{sorted(profiled - recorded)}")

## Step 6 — Run the pipeline over every recording

This is where steps 2, 3 and 4 are actually executed. Everything before this defined
functions; nothing had run on the data yet.

### Structure

`process_recording()` runs the whole chain on one file — load → preprocess → extract features
— tags the resulting epochs with subject, session and case, and returns them together with a
small statistics record. It is written to three rules that make it safe to run in parallel:

- **It takes a plain dict**, not a pandas row, because a dict pickles cheaply and a
  DataFrame row would carry far more baggage to each worker.
- **It prints nothing.** Output from 32 processes at once interleaves into nonsense.
- **It never raises.** A failure is *returned as data* (`ok: False` plus the error message).
  One unreadable file therefore cannot abort a batch of 34.

### Error handling

A failure on one file — missing marker stream, corrupt XDF, unexpected channel set — is
caught, recorded, and the run continues. Failures are listed twice: inline as they are
reported, and again in a block at the end so they cannot be lost in the scroll.

Two quality warnings are also raised per recording: one when **no** epoch survived (almost
always `PTP_THRESH` being too strict for that file), and one when fewer than **50 %** did,
which marks a recording as unusually noisy and worth inspecting in Step 7.

### Running on all CPU cores

The recordings are processed **in parallel**, controlled by `N_JOBS` in the settings cell.
This is worth doing because the work is *embarrassingly parallel*: each recording is a
separate file that is loaded, cleaned and featurised without reference to any other, so
the work divides across cores with no locking, no shared state and no communication —
each worker just returns its finished feature table.

Two details make this work reliably:

- **`joblib` with the `loky` backend.** On Windows, a new process cannot inherit functions
  that were defined in a notebook cell, and the standard `multiprocessing` module fails to
  pickle them. `loky` serialises with `cloudpickle`, which sends the function *by value*,
  so the helpers defined above can stay in this notebook instead of being moved to a
  separate importable module.
- **The worker prints nothing.** Output from several processes at once would interleave
  into nonsense. Each worker returns its log as data, and this cell prints the collected
  results afterwards in catalogue order — so the output is identical and reproducible
  no matter how many cores ran it, or in what order they happened to finish.

Set `N_JOBS = 1` to fall back to a plain sequential loop; results are identical either way,
but exceptions then come with a normal traceback, which is easier to debug.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 6]  Process every recording (in parallel across CPU cores)
# ══════════════════════════════════════════════════════════════════════════════
import time


def process_recording(rec: dict) -> dict:
    """
    Run the whole pipeline on ONE recording. Executed inside a worker process.

    This function is deliberately self-contained and silent:
      * it takes a plain dict (picklable) rather than a pandas row,
      * it prints nothing, because output from parallel workers would interleave,
      * it never raises — a failure is returned as data so one broken file cannot
        bring down the batch.

    Returns
    -------
    dict with keys:
        ok      : bool                — did it work?
        rec     : dict                — the catalogue entry, echoed back
        feat_df : pd.DataFrame | None — the epoch feature table
        log     : dict | None         — per-recording statistics
        error   : str | None          — "ExceptionType: message" when ok is False
    """
    try:
        # ── load -> clean -> epoch -> features ────────────────────────────────
        raw_df   = load_xdf(rec["path"], CH_COLS)
        clean_df = preprocess(raw_df)
        feat_df  = extract_features(clean_df)

        # ── tag every epoch with where it came from ───────────────────────────
        feat_df["subject"] = rec["subject"]
        feat_df["session"] = rec["session"]
        feat_df["case"]    = rec["case"]

        # ── statistics for the summary table ──────────────────────────────────
        # epochs_max is how many complete 2 s windows the recording contained;
        # comparing it with epochs_kept shows how many the PTP filter rejected.
        n_windows = int(len(clean_df) // EPOCH_N)
        n_kept    = len(feat_df)

        log = {
            "subject":     rec["subject"],
            "session":     rec["session"],
            "case":        rec["case"],
            "duration_s":  round(len(clean_df) / SFREQ, 1),
            "samples":     len(clean_df),
            "epochs_kept": n_kept,
            "epochs_max":  n_windows,
            "kept_pct":    round(100 * n_kept / n_windows, 1) if n_windows else 0.0,
            "markers":     sorted(feat_df["marker"].unique().tolist()) if n_kept else [],
        }

        return {"ok": True, "rec": rec, "feat_df": feat_df, "log": log, "error": None}

    except Exception as exc:
        return {"ok": False, "rec": rec, "feat_df": None, "log": None,
                "error": f"{type(exc).__name__}: {exc}"}


# ── Dispatch ──────────────────────────────────────────────────────────────────
# to_dict("records") turns the catalogue into a list of plain dicts, which pickle
# cleanly and cheaply — a DataFrame row would carry far more baggage to each worker.
jobs = catalogue.to_dict("records")

n_workers = cpu_count() if N_JOBS == -1 else N_JOBS
print(f"[STEP 6] Processing {len(jobs)} recording(s) on {n_workers} core(s)...")
t_start = time.time()

if N_JOBS == 1:
    # Sequential path — identical results, but exceptions keep their traceback.
    results = [process_recording(job) for job in jobs]
else:
    # loky is joblib's default backend: separate processes (so the GIL is not a
    # limit) and cloudpickle serialisation (so these notebook-defined functions
    # can be sent to the workers). It also pins each worker's BLAS to one thread,
    # which stops NumPy from oversubscribing the CPU with nested threads.
    results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=0)(
        delayed(process_recording)(job) for job in jobs
    )

elapsed = time.time() - t_start

# ── Report, in catalogue order ────────────────────────────────────────────────
# Workers finish in whatever order they finish; `results` preserves the order the
# jobs were submitted in, so this printout is identical on every run.
all_epochs = []    # feature tables, one per successfully processed recording
failures   = []    # (name, error message) for anything that could not be processed
log_rows   = []    # per-recording statistics, shown as a table in the next cell

print()
for i, res in enumerate(results):
    rec = res["rec"]
    tag = f"{rec['subject']}/{rec['session']}/Case{rec['case']}"

    if not res["ok"]:
        print(f"  [{i + 1:>2}/{len(jobs)}] {tag:<32}  FAILED: {res['error']}")
        failures.append((rec["name"], res["error"]))
        continue

    log = res["log"]
    all_epochs.append(res["feat_df"])
    log_rows.append(log)

    print(f"  [{i + 1:>2}/{len(jobs)}] {tag:<32}  {log['duration_s']:>6.0f}s  ->  "
          f"{log['epochs_kept']:>4} epochs ({log['kept_pct']:>5.1f}% kept)")

    if log["epochs_kept"] == 0:
        print(f"       ! WARNING  no epoch survived — PTP_THRESH "
              f"({PTP_THRESH} uV) may be too strict for this recording.")
    elif log["kept_pct"] < 50:
        print(f"       ! WARNING  only {log['kept_pct']}% of windows survived — "
              f"this recording was unusually noisy.")

print(f"\n[STEP 6] Done in {elapsed:.1f}s — "
      f"{len(all_epochs)} succeeded, {len(failures)} failed.")

if failures:
    print("\n  Failed recordings:")
    for name, err in failures:
        print(f"    {name}\n      {err}")

In [ ]:
# ── [STEP 6b] Per-recording summary table ─────────────────────────────────────
# Scan the kept_pct column: a recording well below the others had a noisy session
# (loose electrode, movement) and may deserve a look before it enters a model.

log_df = pd.DataFrame(log_rows)
if not log_df.empty:
    display(log_df.sort_values(["case", "subject", "session"]).reset_index(drop=True))
    print(f"  Total epochs kept : {log_df['epochs_kept'].sum():,} "
          f"of {log_df['epochs_max'].sum():,} possible "
          f"({100 * log_df['epochs_kept'].sum() / log_df['epochs_max'].sum():.1f}%)")

## Step 7 — Visual check: the signal before and after preprocessing

The summary table says *how many* epochs survived. This step shows **why** — the raw signal
as it came off the headband and the cleaned signal, drawn against **time in seconds** so
they line up sample for sample.

Each figure is one recording: four rows (one per electrode), two columns.

| Column | Signal |
|---|---|
| **BEFORE** | Straight from the XDF, exactly as the Muse recorded it |
| **AFTER** | All six preprocessing stages applied |

The background shading marks the protocol blocks, labelled with what the subject was doing
(from `MARKER_NAMES`), so a disturbance can be traced to the moment it happened.

**The two columns are deliberately not drawn on a shared y-axis.** Raw Muse data sits on a
large DC offset and contains ±1000 µV sentinel spikes, so a shared scale would squash the
cleaned signal into a flat line. Each panel is autoscaled and its actual peak-to-peak range
is printed in the corner — that number is the honest comparison.

### What to look for

- **Before:** a large vertical offset, slow wandering drift, and hard vertical spikes to
  ±1000 µV where the headband dropped samples.
- **After:** centred on zero, drift gone, spikes replaced by interpolated signal, amplitude
  down to roughly ±100 µV.
- **A bad recording** stays visibly wild on the right too — that is what the low `kept_pct`
  values in the table above look like as a picture.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 7]  Plot the raw and cleaned signal against time
# ══════════════════════════════════════════════════════════════════════════════

def select_plot_recordings(catalogue: pd.DataFrame, which) -> list:
    """Turn the PLOT_WHICH setting into a concrete list of catalogue entries."""
    if which == "none" or which is None:
        return []
    if which == "all":
        return catalogue.to_dict("records")
    if which == "first_per_case":
        return catalogue.groupby("case", group_keys=False).head(1).to_dict("records")
    if isinstance(which, int):
        return catalogue.head(which).to_dict("records")
    if isinstance(which, list):
        # Hand-picked: each dict is a set of column filters to match.
        picked = []
        for spec in which:
            match = catalogue
            for key, value in spec.items():
                match = match[match[key] == value]
            if match.empty:
                print(f"  ! WARNING  no recording matches {spec}")
            picked.extend(match.to_dict("records"))
        return picked
    raise ValueError(f"PLOT_WHICH not understood: {which!r}")


def shade_marker_blocks(ax, time_s: np.ndarray, markers: np.ndarray, case: int) -> dict:
    """
    Shade the background of `ax` by protocol block and return {marker: colour}
    so the caller can build a legend.

    A block is a contiguous run of one marker value; the boundaries are found the
    same way as in extract_epochs, so the shading matches how the epochs were cut.
    """
    codes  = sorted(pd.unique(markers).tolist())
    palette = {code: plt.cm.tab10(i % 10) for i, code in enumerate(codes)}

    # Indices where the marker value changes -> block boundaries.
    changes = np.flatnonzero(np.diff(markers) != 0) + 1
    starts  = np.concatenate([[0], changes])
    ends    = np.concatenate([changes, [len(markers)]])

    for start, end in zip(starts, ends):
        ax.axvspan(time_s[start], time_s[end - 1],
                   color=palette[markers[start]], alpha=0.10, zorder=0, lw=0)

    return palette


def plot_before_after(rec: dict, window_s=PLOT_WINDOW_S):
    """
    Draw one recording's raw and cleaned signal side by side, with time on the x-axis.

    The recording is re-loaded and re-cleaned here. Step 6's workers returned only the
    feature tables — keeping every raw and cleaned recording in memory just in case a
    plot was wanted would cost gigabytes, and re-doing the work for a handful of
    selected recordings takes well under a second each.
    """
    raw_df   = load_xdf(rec["path"], CH_COLS)
    clean_df = preprocess(raw_df)

    # ── Restrict to the requested time window ─────────────────────────────────
    # preprocess() never adds or drops rows, so raw and clean share one time axis
    # and a single mask applies to both.
    time_all = raw_df["time_s"].values
    if window_s is None:
        mask = np.ones(len(time_all), dtype=bool)
    else:
        mask = (time_all >= window_s[0]) & (time_all <= window_s[1])
        if not mask.any():
            print(f"  ! WARNING  {rec['name']}: nothing recorded in window {window_s} s "
                  f"(recording spans {time_all[0]:.0f}-{time_all[-1]:.0f} s) — skipped.")
            return

    time_s  = time_all[mask]
    markers = raw_df["marker"].values[mask].astype(int)

    # ── Decimate for display when there are too many points to draw ───────────
    stride = max(1, len(time_s) // PLOT_MAX_POINTS)

    # ── Figure: one row per channel, BEFORE | AFTER ───────────────────────────
    n_ch = len(CH_COLS)
    fig, axes = plt.subplots(n_ch, 2, figsize=(15, 2.2 * n_ch), sharex=True)

    span_txt = ("whole recording" if window_s is None
                else f"{window_s[0]:.0f}-{window_s[1]:.0f} s")
    fig.suptitle(
        f"Preprocessing check — {rec['subject']} / {rec['session']} / Case {rec['case']}\n"
        f"{span_txt}"
        + (f"   (displayed every {stride}th sample)" if stride > 1 else "")
        + f"   |   {len(time_s) / SFREQ:.0f} s shown at {SFREQ:.0f} Hz",
        fontsize=12, fontweight="bold",
    )

    columns = [
        (raw_df,   "BEFORE  —  raw signal from the XDF",                    "#3B6EA5"),
        (clean_df, f"AFTER  —  bandpass {BANDPASS_LO:g}-{BANDPASS_HI:g} Hz, notch "
                   f"{LINE_FREQ} Hz, >{THRESHOLD} uV interpolated, re-referenced",
                   "#B05A2F"),
    ]

    palette = {}
    for col_i, (source_df, title, colour) in enumerate(columns):
        for ch_i, channel in enumerate(CH_COLS):
            ax     = axes[ch_i, col_i]
            signal = source_df[channel].values[mask]

            # Shade the protocol blocks behind the trace.
            palette = shade_marker_blocks(ax, time_s, markers, rec["case"])

            ax.plot(time_s[::stride], signal[::stride],
                    color=colour, lw=0.5, alpha=0.9)

            # Peak-to-peak range: the honest scale comparison between the columns,
            # since the two are autoscaled independently.
            ptp = float(np.ptp(signal))
            ax.text(0.995, 0.94, f"peak-to-peak {ptp:,.0f} uV",
                    transform=ax.transAxes, ha="right", va="top", fontsize=7.5,
                    bbox=dict(boxstyle="round,pad=0.25", fc="white", alpha=0.75, lw=0))

            ax.set_ylabel(f"{channel}\n(uV)", fontsize=9)
            ax.grid(True, alpha=0.22, ls="--")
            ax.tick_params(labelsize=8)
            if ch_i == 0:
                ax.set_title(title, fontsize=10, fontweight="bold")
            if ch_i == n_ch - 1:
                ax.set_xlabel("Time since start of recording (s)", fontsize=10)

    # ── Legend naming each shaded protocol block ──────────────────────────────
    names   = MARKER_NAMES.get(rec["case"], {})
    handles = [
        mpatches.Patch(color=colour, alpha=0.35,
                       label=f"{code} — {names.get(code, 'unknown')}")
        for code, colour in sorted(palette.items())
    ]
    fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 7),
               fontsize=8.5, title="Protocol block (marker)", frameon=False,
               bbox_to_anchor=(0.5, -0.04))

    plt.tight_layout(rect=[0, 0.02, 1, 0.97])

    if PLOT_SAVE:
        PLOT_PATH.mkdir(parents=True, exist_ok=True)
        out_file = PLOT_PATH / (f"{rec['subject']}_{rec['session']}_Case{rec['case']}"
                                f"_before_after.png")
        fig.savefig(out_file, dpi=110, bbox_inches="tight")

    plt.show()


# ── Run the visual check ──────────────────────────────────────────────────────
PLOT_PATH = PROJECT_ROOT / PLOT_DIR

if not PLOT_BEFORE_AFTER:
    print("[STEP 7] Skipped (PLOT_BEFORE_AFTER is False).")
else:
    to_plot = select_plot_recordings(catalogue, PLOT_WHICH)
    print(f"[STEP 7] Plotting {len(to_plot)} recording(s)  "
          f"(PLOT_WHICH = {PLOT_WHICH!r}, window = {PLOT_WINDOW_S})\n")

    for rec in to_plot:
        plot_before_after(rec)

    if PLOT_SAVE and to_plot:
        print(f"\n[STEP 7] Figures saved to {PLOT_PATH}")

## Step 8 — The Welch transform: periodograms and feature vectors

Step 7 showed the signal in **time**. This step shows the same recordings in **frequency** —
how the Welch transform turns each cleaned epoch into the 20 numbers written to the CSV.

### The problem Welch solves

A recording of brain activity is a wave that changes over time. We want to know **which
rhythms it is made of** — how much 10 Hz alpha, how much 6 Hz theta — because those
amounts are the features the classifier learns from.

The textbook tool for that is the Fourier transform: feed it a signal, get back how much
power sits at each frequency. Applied directly to one 2-second epoch, though, it produces a
badly unreliable answer. EEG is dominated by noise, and a single transform faithfully
reports **that particular noise** along with the signal. Run it on the next epoch and the
spectrum looks quite different, even though the brain state has not changed. The estimate is
correct on average but has enormous variance — and a feature that jumps around at random is
worthless to a model.

### How Welch's method fixes it

Welch's insight is to trade a little frequency detail for a lot of stability. Instead of one
transform of the whole epoch, it takes **several transforms of shorter, overlapping pieces
and averages them**. Averaging cancels the random part while the consistent part survives:

```
  one 2-second epoch (512 samples)
  |-------------------------------------------------------|
  [ segment 1: 1 s, 256 samples ]                            -> taper -> FFT -> power  \
                  [ segment 2: 1 s ]                         -> taper -> FFT -> power   >- average
                                  [ segment 3: 1 s ]         -> taper -> FFT -> power  /
                                                                                          |
   50 % overlap, so no part of the epoch is under-used                                    v
                                                                             a stable spectrum (PSD)
```

Three things are happening, and each has a reason:

1. **Split into 1-second segments.** More segments to average means a steadier result. The
   cost is resolution: a 1-second segment can only distinguish frequencies 1 Hz apart, where
   the full 2 seconds could have resolved 0.5 Hz. For band power spanning several Hz that
   trade is clearly worth it.
2. **Taper each segment with a Hann window.** A segment cut out of a longer recording starts
   and ends abruptly, and the transform interprets those artificial edges as a burst of
   energy smeared across every frequency — *spectral leakage*, which would let a strong
   delta wave contaminate the alpha measurement. The Hann window fades each segment in and
   out so there are no sharp edges.
3. **Overlap the segments by 50 %.** Tapering suppresses the data near each segment's edges,
   so overlapping ensures every part of the epoch sits in the middle of some segment and
   contributes fully.

The result is a **power spectral density** (PSD) in µV²/Hz: power per unit of frequency.
Band power is then the **area under that curve** between the band edges — computed by
trapezoidal integration, which just sums up thin rectangular slices under the curve. One
number per channel per band, so 4 × 5 = **20 features per epoch**.

### The four figures

| Figure | Shows | Welch window |
|---|---|---|
| **W** | **How the method works**, step by step on one real epoch | 1 s → 1 Hz resolution |
| **A** | Full-recording spectrum, **before vs after** preprocessing | 4 s → 0.25 Hz resolution |
| **B** | **Every accepted epoch's** PSD, averaged per protocol block | 1 s → 1 Hz resolution |
| **C** | The resulting **feature vectors** — what the CSV actually contains | — |

Figure B and the features come from the **same `welch_psd()` call**, so the shaded areas in
that figure are literally the numbers in your CSV, not a re-creation of them.

### What to look for

- **Figure A:** the "after" curve should fall off sharply below 1 Hz and above 50 Hz (the
  bandpass), with a narrow notch at 50 Hz. The huge low-frequency ramp in "before" is the
  DC offset and drift being removed.
- **Figure B:** an alpha bump around 8–13 Hz that is **stronger during eyes-closed** than
  eyes-open is the classic sanity check that the electrodes were working and the markers
  are aligned to the right blocks.
- **Figure C:** delta is normally the largest band and gamma the smallest, because EEG power
  falls off roughly as 1/f. Frontal channels (AF7/AF8) usually carry more power than the
  mastoids.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 8]  Welch periodograms and feature vectors
# ══════════════════════════════════════════════════════════════════════════════

def shade_bands(ax):
    """Tint the frequency axis with one colour band per entry in BANDS."""
    for band, (lo, hi) in BANDS.items():
        ax.axvspan(lo, hi, alpha=0.13, color=BAND_COLORS[band], zorder=0, lw=0)


def band_legend_handles():
    """Legend patches naming each frequency band and its edges."""
    return [mpatches.Patch(color=BAND_COLORS[b], alpha=0.55, label=f"{b} ({lo:g}-{hi:g} Hz)")
            for b, (lo, hi) in BANDS.items()]


def save_fig(fig, rec: dict, suffix: str):
    """Write a figure to PLOT_PATH using a consistent name."""
    if not PLOT_SAVE:
        return
    PLOT_PATH.mkdir(parents=True, exist_ok=True)
    fig.savefig(PLOT_PATH / f"{rec['subject']}_{rec['session']}_Case{rec['case']}_{suffix}.png",
                dpi=110, bbox_inches="tight")


def plot_welch_walkthrough(clean_df: pd.DataFrame, rec: dict):
    """
    Figure W — take one real epoch and show every stage of Welch's method on it.

    Panels:
      1. the epoch in the time domain, with the overlapping segments marked
      2. the Hann taper, and what it does to one segment
      3. each segment's own spectrum, and their average
      4. a single whole-epoch transform vs the Welch average (the variance argument)
      5. the final PSD, with each band's area shaded and its integral labelled
    """
    channel = PSD_DEMO_CHANNEL if PSD_DEMO_CHANNEL in CH_COLS else CH_COLS[0]
    ch_i    = CH_COLS.index(channel)

    epochs = extract_epochs(clean_df)
    if not epochs:
        print("  (no epoch survived — figure W skipped)")
        return

    # Use a representative epoch rather than an arbitrary one: the epoch whose total
    # variance is closest to the median, so the illustration is neither the quietest
    # nor the noisiest window in the recording.
    variances = np.array([ep["data"][:, ch_i].var() for ep in epochs])
    epoch     = epochs[int(np.argsort(variances)[len(variances) // 2])]
    signal    = epoch["data"][:, ch_i]
    n         = len(signal)

    nperseg  = int(min(WELCH_NPERSEG, n))
    noverlap = nperseg // 2
    step     = nperseg - noverlap
    starts   = list(range(0, n - nperseg + 1, step))
    taper    = np.hanning(nperseg)
    time_ax  = np.arange(n) / SFREQ

    fig = plt.figure(figsize=(16, 15))
    grid = fig.add_gridspec(4, 2, height_ratios=[1.0, 1.0, 1.1, 1.2], hspace=0.42, wspace=0.22)
    fig.suptitle(
        f"[W] How Welch's method turns one epoch into {len(BANDS)} numbers\n"
        f"{rec['subject']} / {rec['session']} / Case {rec['case']}  —  channel {channel}, "
        f"one {EPOCH_S:g} s epoch from block "
        f"{epoch['marker']} ({MARKER_NAMES.get(rec['case'], {}).get(epoch['marker'], '?')})",
        fontsize=13, fontweight="bold")

    seg_colors = plt.cm.viridis(np.linspace(0.15, 0.8, len(starts)))

    # ── Panel 1: the epoch, and how it is cut into overlapping segments ───────
    ax1 = fig.add_subplot(grid[0, :])
    ax1.plot(time_ax, signal, color="#333333", lw=0.8)
    ax1.set_title(f"1.  The epoch: {EPOCH_S:g} s of cleaned EEG ({n} samples), cut into "
                  f"{len(starts)} overlapping segments of {nperseg / SFREQ:g} s",
                  fontsize=10, fontweight="bold", loc="left")
    ax1.set_xlabel("Time within the epoch (s)", fontsize=9)
    ax1.set_ylabel(f"{channel} (uV)", fontsize=9)
    ax1.grid(True, alpha=0.25, ls="--")

    # Draw each segment as a labelled bracket beneath the trace.
    y_lo, y_hi = ax1.get_ylim()
    ax1.set_ylim(y_lo - 0.42 * (y_hi - y_lo), y_hi)
    for s_i, start in enumerate(starts):
        t0, t1 = start / SFREQ, (start + nperseg) / SFREQ
        y_bar  = y_lo - (0.10 + 0.10 * s_i) * (y_hi - y_lo)
        ax1.axvspan(t0, t1, color=seg_colors[s_i], alpha=0.10, lw=0)
        ax1.plot([t0, t1], [y_bar, y_bar], color=seg_colors[s_i], lw=4, solid_capstyle="butt")
        ax1.text((t0 + t1) / 2, y_bar, f" segment {s_i + 1} ", color="white", fontsize=8,
                 fontweight="bold", ha="center", va="center")
    ax1.text(0.995, 0.97, f"neighbouring segments overlap by {100 * noverlap / nperseg:.0f}%",
             transform=ax1.transAxes, ha="right", va="top", fontsize=8.5, style="italic")

    # ── Panel 2: the Hann taper and its effect on one segment ────────────────
    ax2 = fig.add_subplot(grid[1, 0])
    seg_raw = signal[starts[0]:starts[0] + nperseg]
    seg_t   = np.arange(nperseg) / SFREQ
    ax2.plot(seg_t, seg_raw, color="#999999", lw=0.8, label="segment 1, as cut")
    ax2.plot(seg_t, seg_raw * taper, color=seg_colors[0], lw=1.0, label="after the Hann taper")
    ax2.set_title("2.  Taper each segment: fade the edges to zero", fontsize=10,
                  fontweight="bold", loc="left")
    ax2.set_xlabel("Time within the segment (s)", fontsize=9)
    ax2.set_ylabel(f"{channel} (uV)", fontsize=9)
    ax2.legend(fontsize=8, loc="upper right")
    ax2.grid(True, alpha=0.25, ls="--")
    # The window shape itself, on a twin axis so its 0-1 scale is readable.
    ax2_w = ax2.twinx()
    ax2_w.plot(seg_t, taper, color="#B05A2F", lw=1.6, ls="--", alpha=0.85)
    ax2_w.set_ylabel("Hann window", fontsize=8.5, color="#B05A2F")
    ax2_w.tick_params(axis="y", labelsize=7.5, colors="#B05A2F")
    ax2_w.set_ylim(0, 1.05)
    ax2.text(0.5, 0.02,
             "without this, the abrupt cut leaks energy across all frequencies",
             transform=ax2.transAxes, ha="center", va="bottom", fontsize=8, style="italic")

    # ── Panel 3: each segment's spectrum, and the average ─────────────────────
    ax3 = fig.add_subplot(grid[1, 1])
    seg_psds = []
    for s_i, start in enumerate(starts):
        seg = signal[start:start + nperseg]
        f_seg, p_seg = welch(seg, fs=SFREQ, nperseg=nperseg, noverlap=0,
                             window="hann", scaling="density")
        seg_psds.append(p_seg)
        ax3.semilogy(f_seg, p_seg, color=seg_colors[s_i], lw=1.0, alpha=0.85,
                     label=f"segment {s_i + 1}")
    mean_psd = np.mean(seg_psds, axis=0)
    ax3.semilogy(f_seg, mean_psd, color="#C44E52", lw=2.4, label="their average")
    ax3.set_title("3.  Transform each segment, then average", fontsize=10,
                  fontweight="bold", loc="left")
    ax3.set_xlabel("Frequency (Hz)", fontsize=9)
    ax3.set_ylabel("Power (uV^2/Hz)", fontsize=9)
    ax3.set_xlim(0, min(PSD_FMAX, 52))
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.25, ls="--")

    # ── Panel 4: one transform of the whole epoch vs the Welch average ───────
    ax4 = fig.add_subplot(grid[2, :])
    f_one, p_one = welch(signal, fs=SFREQ, nperseg=n, noverlap=0,
                         window="hann", scaling="density")
    f_welch, p_welch = welch_psd(epoch["data"], WELCH_NPERSEG)
    ax4.semilogy(f_one, p_one, color="#999999", lw=0.9,
                 label=f"one transform of the whole epoch  ({SFREQ / n:.2f} Hz resolution, jagged)")
    ax4.semilogy(f_welch, p_welch[ch_i], color="#C44E52", lw=2.2,
                 label=f"Welch: {len(starts)} segments averaged  "
                       f"({SFREQ / nperseg:.0f} Hz resolution, stable)")
    ax4.set_title("4.  Why bother: a single transform is jagged and unrepeatable, "
                  "the average is smooth", fontsize=10, fontweight="bold", loc="left")
    ax4.set_xlabel("Frequency (Hz)", fontsize=9)
    ax4.set_ylabel("Power (uV^2/Hz)", fontsize=9)
    ax4.set_xlim(0, min(PSD_FMAX, 52))
    ax4.legend(fontsize=8.5, loc="upper right")
    ax4.grid(True, alpha=0.25, ls="--")

    # ── Panel 5: integrate each band -> the feature values ───────────────────
    ax5 = fig.add_subplot(grid[3, :])
    shade_bands(ax5)
    ax5.plot(f_welch, p_welch[ch_i], color="#333333", lw=1.6, zorder=5)
    ax5.set_yscale("log")
    # Headroom above the curve so the labels below sit clear of it.
    ax5.set_ylim(top=ax5.get_ylim()[1] * 60)

    # Label positions use a blended transform: x in data units (so a label sits over
    # its band) and y as a fraction of the axes height (so a label can never be pushed
    # off the top by a tall peak, which is easy to do on a log axis). Heights alternate
    # to keep neighbouring labels from colliding in the narrow bands.
    label_heights = [0.93, 0.79]
    for band_i, (band, (lo, hi)) in enumerate(BANDS.items()):
        idx = np.flatnonzero((f_welch >= lo) & (f_welch <= hi))
        if not len(idx):
            continue
        ax5.fill_between(f_welch[idx], p_welch[ch_i, idx], alpha=0.55,
                         color=BAND_COLORS[band], zorder=3)
        power = TRAPZ(p_welch[ch_i, idx], f_welch[idx])
        ax5.text((lo + hi) / 2, label_heights[band_i % len(label_heights)],
                 f"{channel}_{band}\n= {power:,.1f}",
                 transform=ax5.get_xaxis_transform(),
                 fontsize=8.5, ha="center", va="top", fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=BAND_COLORS[band],
                           alpha=0.92, lw=1.2), zorder=6)
    ax5.set_title(f"5.  The area under the curve inside each band = the feature value "
                  f"(uV^2/Hz).  These {len(BANDS)} numbers are {channel}'s columns in the CSV",
                  fontsize=10, fontweight="bold", loc="left")
    ax5.set_xlabel("Frequency (Hz)", fontsize=9)
    ax5.set_ylabel("Power (uV^2/Hz)", fontsize=9)
    ax5.set_xlim(0, min(PSD_FMAX, 52))
    ax5.grid(True, alpha=0.25, ls="--")
    ax5.legend(handles=band_legend_handles(), fontsize=8, loc="lower left", ncol=len(BANDS))

    plt.tight_layout(rect=[0, 0, 1, 0.955])
    save_fig(fig, rec, "W_welch_walkthrough")
    plt.show()

    # The averaged segment spectra in panel 3 and the Welch output in panel 4 are the
    # same computation; confirm rather than assert it, so a mismatch is visible but
    # never stops the notebook.
    agreement = np.allclose(mean_psd, p_welch[ch_i], rtol=1e-6, atol=1e-12)
    print(f"  panel 3 average == panel 4 Welch output: {agreement}")


def plot_periodograms(rec: dict):
    """Draw figures W, A, B and C for one recording (whichever are listed in PSD_FIGURES)."""
    # Re-loaded rather than cached: holding every raw + cleaned recording in memory
    # would cost gigabytes, and redoing this takes well under a second.
    raw_df   = load_xdf(rec["path"], CH_COLS)
    clean_df = preprocess(raw_df)
    n_ch     = len(CH_COLS)
    names    = MARKER_NAMES.get(rec["case"], {})
    title_id = f"{rec['subject']} / {rec['session']} / Case {rec['case']}"

    # ══ FIGURE W — the method walk-through ═══════════════════════════════════
    # Drawn only for the first recording: it explains how Welch's method works,
    # which is the same story for every file.
    if "W" in PSD_FIGURES and rec is FIRST_PLOT_REC:
        plot_welch_walkthrough(clean_df, rec)

    # ══ FIGURE A — full-recording spectrum, before vs after ═══════════════════
    if "A" in PSD_FIGURES:
        freqs_raw,   psd_raw   = welch_psd(raw_df[CH_COLS].values.astype(float),
                                           PSD_RECORDING_NPERSEG)
        freqs_clean, psd_clean = welch_psd(clean_df[CH_COLS].values.astype(float),
                                           PSD_RECORDING_NPERSEG)

        figA, axesA = plt.subplots(n_ch, 2, figsize=(14, 2.5 * n_ch), sharex=True)
        figA.suptitle(
            f"[A] Periodogram before vs after preprocessing — {title_id}\n"
            f"Welch, Hann window {PSD_RECORDING_NPERSEG / SFREQ:.0f} s, 50% overlap "
            f"-> {SFREQ / PSD_RECORDING_NPERSEG:.2f} Hz resolution",
            fontsize=12, fontweight="bold")

        columns = [
            (freqs_raw,   psd_raw,   "#3B6EA5", "BEFORE  —  raw signal from the XDF"),
            (freqs_clean, psd_clean, "#B05A2F",
             f"AFTER  —  bandpass {BANDPASS_LO:g}-{BANDPASS_HI:g} Hz + notch {LINE_FREQ} Hz"),
        ]
        for col_i, (freqs, psd, colour, title) in enumerate(columns):
            keep = freqs <= PSD_FMAX
            for ch_i, channel in enumerate(CH_COLS):
                ax = axesA[ch_i, col_i]
                shade_bands(ax)
                # Log y: EEG power spans several orders of magnitude across the
                # spectrum, so a linear axis would show only the delta peak.
                ax.semilogy(freqs[keep], psd[ch_i, keep], color=colour, lw=0.9, alpha=0.9)
                ax.axvline(LINE_FREQ, color="grey", ls=":", lw=1.0, alpha=0.7)
                ax.set_ylabel(f"{channel}\n(uV^2/Hz)", fontsize=9)
                ax.set_xlim(0, PSD_FMAX)
                ax.grid(True, alpha=0.22, ls="--")
                ax.tick_params(labelsize=8)
                if ch_i == 0:
                    ax.set_title(title, fontsize=10, fontweight="bold")
                if ch_i == n_ch - 1:
                    ax.set_xlabel("Frequency (Hz)", fontsize=10)

        figA.legend(handles=band_legend_handles(), loc="lower center",
                    ncol=len(BANDS), fontsize=8.5, title="Frequency bands",
                    frameon=False, bbox_to_anchor=(0.5, -0.04))
        plt.tight_layout(rect=[0, 0.02, 1, 0.97])
        save_fig(figA, rec, "A_periodogram_before_after")
        plt.show()

    # ── Epochs are needed for both B and C ────────────────────────────────────
    if not ({"B", "C"} & set(PSD_FIGURES)):
        return

    epochs = extract_epochs(clean_df)
    if not epochs:
        print(f"  ({title_id}: no epoch survived — figures B and C skipped)")
        return

    # Group the epoch PSDs by marker so each protocol block gets its own mean curve.
    psd_by_marker, freqs_ep = {}, None
    for epoch in epochs:
        freqs_ep, psd = welch_psd(epoch["data"], WELCH_NPERSEG)
        psd_by_marker.setdefault(epoch["marker"], []).append(psd)
    psd_by_marker = {mk: np.array(v) for mk, v in psd_by_marker.items()}   # (n_ep, n_ch, n_f)

    markers  = sorted(psd_by_marker)
    mk_color = {mk: plt.cm.tab10(i % 10) for i, mk in enumerate(markers)}
    psd_mean = np.concatenate(list(psd_by_marker.values()), axis=0).mean(axis=0)

    # ══ FIGURE B — epoch-level periodogram ═══════════════════════════════════
    if "B" in PSD_FIGURES:
        figB, axesB = plt.subplots(n_ch, 1, figsize=(12, 2.7 * n_ch), sharex=True)
        figB.suptitle(
            f"[B] Epoch-level Welch periodogram — {title_id}\n"
            f"{len(epochs)} epochs x {EPOCH_S:g} s | Hann {WELCH_NPERSEG / SFREQ:.0f} s window "
            f"-> {SFREQ / WELCH_NPERSEG:.0f} Hz resolution\n"
            f"Thin line = one epoch | Thick = mean per block | "
            f"Filled area under the mean = the band-power feature",
            fontsize=11, fontweight="bold")

        keep = freqs_ep <= min(PSD_FMAX, 52)
        for ch_i, channel in enumerate(CH_COLS):
            ax = axesB[ch_i]
            shade_bands(ax)

            for mk in markers:
                stack = psd_by_marker[mk]
                for epoch_psd in stack[:, ch_i, :]:
                    ax.semilogy(freqs_ep[keep], epoch_psd[keep],
                                color=mk_color[mk], lw=0.3, alpha=0.15)
                ax.semilogy(freqs_ep[keep], stack.mean(axis=0)[ch_i, keep],
                            color=mk_color[mk], lw=2.1, alpha=0.95,
                            label=f"{mk} — {names.get(mk, 'unknown')}  (n={len(stack)})")

            # Fill each band under the overall mean and print the integrated power:
            # this is exactly the number band_power() writes into the CSV.
            for band, (lo, hi) in BANDS.items():
                idx = np.flatnonzero((freqs_ep >= lo) & (freqs_ep <= hi))
                if not len(idx):
                    continue
                ax.fill_between(freqs_ep[idx], psd_mean[ch_i, idx],
                                alpha=0.40, color=BAND_COLORS[band])
                ax.annotate(f"{band}\n{TRAPZ(psd_mean[ch_i, idx], freqs_ep[idx]):.1f}",
                            xy=((lo + hi) / 2, psd_mean[ch_i, idx].max()),
                            fontsize=6.5, ha="center", va="bottom", fontweight="bold",
                            bbox=dict(boxstyle="round,pad=0.15", fc="white",
                                      alpha=0.75, lw=0))

            ax.set_ylabel(f"{channel}\n(uV^2/Hz)", fontsize=9)
            ax.set_xlim(0, min(PSD_FMAX, 52))
            ax.grid(True, alpha=0.22, ls="--")
            ax.tick_params(labelsize=8)
            if ch_i == 0:
                ax.legend(fontsize=8, loc="upper right", title="Protocol block")
            if ch_i == n_ch - 1:
                ax.set_xlabel("Frequency (Hz)", fontsize=10)

        plt.tight_layout(rect=[0, 0, 1, 0.96])
        save_fig(figB, rec, "B_epoch_periodogram")
        plt.show()

    # ══ FIGURE C — the resulting feature vectors ═════════════════════════════
    if "C" in PSD_FIGURES:
        feat_df   = extract_features(clean_df)
        bar_color = [BAND_COLORS[c.rsplit("_", 1)[1]] for c in FEAT_COLS]

        figC, axesC = plt.subplots(1, 3, figsize=(21, 5.2))
        figC.suptitle(
            f"[C] Welch band-power feature vectors — {title_id}   ({len(feat_df)} epochs)\n"
            f"These {len(FEAT_COLS)} numbers per epoch are what the CSV contains",
            fontsize=12, fontweight="bold")

        # Panel 1 — the mean feature vector across every epoch, +/- 1 std.
        means, stds = feat_df[FEAT_COLS].mean().values, feat_df[FEAT_COLS].std().values
        axesC[0].bar(range(len(FEAT_COLS)), means, yerr=stds, color=bar_color,
                     alpha=0.8, edgecolor="white",
                     error_kw={"elinewidth": 0.8, "capsize": 2})
        axesC[0].set_xticks(range(len(FEAT_COLS)))
        axesC[0].set_xticklabels(FEAT_COLS, rotation=90, fontsize=7)
        axesC[0].set_ylabel("Band power (uV^2/Hz)  +/- 1 std", fontsize=9)
        axesC[0].set_title("Mean feature vector (all epochs)", fontsize=10)
        axesC[0].set_yscale("log")     # 1/f means delta dwarfs gamma on a linear axis
        axesC[0].grid(True, alpha=0.3, axis="y", ls="--")
        # Separator between channels, so the 4 blocks of 5 bands are easy to read.
        for x in range(len(BANDS), len(FEAT_COLS), len(BANDS)):
            axesC[0].axvline(x - 0.5, color="grey", lw=0.7, ls="--", alpha=0.6)
        axesC[0].legend(handles=band_legend_handles(), fontsize=7.5, loc="upper right")

        # Panel 2 — mean feature vector per protocol block, z-scored per feature so
        # blocks can be compared despite the 1/f scale difference between bands.
        by_marker = feat_df.groupby("marker")[FEAT_COLS].mean()
        logged    = np.log10(by_marker.values.clip(min=1e-30))
        zscored   = (logged - logged.mean(0)) / (logged.std(0) + 1e-12)
        im1 = axesC[1].imshow(zscored, aspect="auto", cmap="RdYlBu_r", vmin=-2.5, vmax=2.5)
        plt.colorbar(im1, ax=axesC[1], label="z-score of log10 power", fraction=0.04, pad=0.02)
        axesC[1].set_xticks(range(len(FEAT_COLS)))
        axesC[1].set_xticklabels(FEAT_COLS, rotation=90, fontsize=7)
        axesC[1].set_yticks(range(len(by_marker)))
        axesC[1].set_yticklabels([f"{mk} — {names.get(mk, '?')}" for mk in by_marker.index],
                                 fontsize=8)
        axesC[1].set_title("Mean per protocol block\n(red = high for that block, blue = low)",
                           fontsize=10)
        for x in range(len(BANDS), len(FEAT_COLS), len(BANDS)):
            axesC[1].axvline(x - 0.5, color="white", lw=1.5)

        # Panel 3 — every epoch as a column: the raw material the classifier sees.
        sorted_df = feat_df.sort_values("marker")
        matrix    = np.log10(sorted_df[FEAT_COLS].values.clip(min=1e-30))
        im2 = axesC[2].imshow(matrix.T, aspect="auto", cmap="RdYlBu_r", interpolation="nearest")
        plt.colorbar(im2, ax=axesC[2], label="log10 power", fraction=0.04, pad=0.02)
        boundary = 0
        for mk in sorted_df["marker"].unique()[:-1]:
            boundary += int((sorted_df["marker"] == mk).sum())
            axesC[2].axvline(boundary - 0.5, color="white", lw=1.5, ls="--")
        axesC[2].set_yticks(range(len(FEAT_COLS)))
        axesC[2].set_yticklabels(FEAT_COLS, fontsize=7)
        axesC[2].set_xlabel("Epoch (sorted by block; dashed = block boundary)", fontsize=9)
        axesC[2].set_title("Every epoch's feature vector", fontsize=10)
        for y in range(len(BANDS), len(FEAT_COLS), len(BANDS)):
            axesC[2].axhline(y - 0.5, color="white", lw=1.2)

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        save_fig(figC, rec, "C_feature_vectors")
        plt.show()


# ── Run the frequency-domain check ────────────────────────────────────────────
if not PLOT_PERIODOGRAM:
    print("[STEP 8] Skipped (PLOT_PERIODOGRAM is False).")
else:
    to_plot = select_plot_recordings(catalogue, PLOT_WHICH)

    # Figure W is drawn once, for this recording; plot_periodograms checks identity
    # against it. None when nothing was selected.
    FIRST_PLOT_REC = to_plot[0] if to_plot else None

    print(f"[STEP 8] Welch periodograms for {len(to_plot)} recording(s)  "
          f"(figures {', '.join(PSD_FIGURES)})\n")

    for rec in to_plot:
        plot_periodograms(rec)

    if PLOT_SAVE and to_plot:
        print(f"\n[STEP 8] Figures saved to {PLOT_PATH}")

## Step 9 — Assemble and save one CSV per case

### What the code does

Five operations, in order:

1. **Concatenate** every worker's feature table into one long frame.
2. **Join** the Age/Gender lookup on `subject`. A **left** join is used deliberately: it keeps
   every epoch even when a subject is missing from the profiles file, giving them `NaN`
   metadata rather than dropping their data. Step 5 has already warned about anyone affected,
   so the choice is between a visible gap and a silent loss of rows — the gap is safer.
3. **Order the columns** so the file reads sensibly: identity, then metadata, then features.
4. **Sort** by case, subject, session, block, epoch index, so each file reads chronologically
   within a subject rather than in whatever order the parallel workers finished.
5. **Split by case and write.** `float_format="%.6g"` keeps six significant figures, which is
   far beyond the precision of the measurement and roughly halves the file size against
   pandas' default.

**Output:** `01. Data/01. Imported/Case1_epochs.csv` … `Case5_epochs.csv`

| Column | Meaning |
|---|---|
| `subject` | `sub-P001` … — who was recorded |
| `session` | `ses-S001` … — which recording session |
| `case` | 1–5 — which protocol |
| `marker` | Event code in force during this epoch (see the table at the top) |
| `block` | Which contiguous run of that marker, in time order |
| `epoch_idx` | Position of the epoch inside its block (0, 1, 2, …) |
| `Age`, `Gender` | From `Respondents_profiles.csv` |
| `TP9_delta` … `TP10_gamma` | The 20 band-power features, µV²/Hz |

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 9]  Assemble, join metadata, and write one CSV per case
# ══════════════════════════════════════════════════════════════════════════════

if not all_epochs:
    raise RuntimeError("No recording was processed successfully — nothing to save.")

# ── Concatenate every recording into one long table ───────────────────────────
epochs_df = pd.concat(all_epochs, ignore_index=True)

# ── Join Age + Gender on subject ──────────────────────────────────────────────
# A left join keeps every epoch even if the subject is missing from the profiles
# file; those rows simply get NaN metadata (Step 5 already warned about them).
epochs_df = epochs_df.merge(person_meta, on="subject", how="left")

# ── Put the columns in a readable order ───────────────────────────────────────
ordered_cols = LABEL_COLS + META_COLS + FEAT_COLS
epochs_df    = epochs_df[ordered_cols]

# ── Sort so the file reads chronologically within each subject ────────────────
epochs_df = epochs_df.sort_values(
    ["case", "subject", "session", "block", "epoch_idx"]
).reset_index(drop=True)

print(f"[STEP 9] Combined table: {len(epochs_df):,} epochs x {len(epochs_df.columns)} columns\n")

# ── Split by case and write ───────────────────────────────────────────────────
written = []
for case, case_df in epochs_df.groupby("case"):
    out_file = OUT_PATH / OUT_PATTERN.format(case=case)
    case_df.to_csv(out_file, index=False, float_format=FLOAT_FMT)

    size_mb = out_file.stat().st_size / 1e6
    written.append({
        "file":     out_file.name,
        "case":     case,
        "epochs":   len(case_df),
        "subjects": case_df["subject"].nunique(),
        "sessions": case_df.groupby("subject")["session"].nunique().sum(),
        "markers":  sorted(case_df["marker"].unique().tolist()),
        "size_MB":  round(size_mb, 2),
    })
    print(f"  saved  {out_file.name:<22} {len(case_df):>6,} epochs   {size_mb:>6.2f} MB")

print(f"\n[STEP 9] {len(written)} file(s) written to {OUT_PATH}")

## Step 10 — Verify what was written

A last look at the output before the analysis notebooks pick it up: what is in each file,
how the epochs are spread across subjects and conditions, and a preview of the actual rows.

It ends with a **frequency table per case**: one row per recording, one column per protocol
block, showing how many epochs passed artefact rejection — plus each recording's overall
pass rate, so a thin row can immediately be read as "noisy recording" rather than "short
recording".

Worth checking here:
- **Every case has the markers its protocol specifies** (see the table at the top of the notebook).
- **No subject dominates a case** — a badly unbalanced class distribution will bias any classifier trained on it.
- **No block is empty for a recording.** A `0` means that condition contributed nothing for
  that subject, so they cannot be used for any comparison involving it.
- **The feature values look plausible.** Band power in µV²/Hz should fall roughly in the
  1–1000 range, with delta largest and gamma smallest. Exact zeros or astronomically large
  values point at a preprocessing problem.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  [STEP 10]  Verify the output
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 78)
print("  FILES WRITTEN")
print("=" * 78)
display(pd.DataFrame(written))

print("\n" + "=" * 78)
print("  EPOCHS PER SUBJECT x CASE")
print("=" * 78)
display(pd.crosstab(epochs_df["subject"], epochs_df["case"], margins=True, margins_name="TOTAL"))

print("\n" + "=" * 78)
print("  EPOCHS PER MARKER x CASE  (0 = that marker does not occur in that protocol)")
print("=" * 78)
display(pd.crosstab(epochs_df["marker"], epochs_df["case"]))

print("\n" + "=" * 78)
print("  FEATURE VALUE RANGES  (uV^2/Hz, pooled over all epochs)")
print("=" * 78)
display(epochs_df[FEAT_COLS].describe().T[["mean", "std", "min", "50%", "max"]].round(2))

print("\n" + "=" * 78)
print("  PREVIEW  (first 5 rows)")
print("=" * 78)
display(epochs_df.head())


# ══════════════════════════════════════════════════════════════════════════════
#  Frequency table per case: epochs that passed, by recording x protocol block
# ══════════════════════════════════════════════════════════════════════════════

def epochs_by_recording(case: int) -> pd.DataFrame:
    """
    Crosstab of accepted epochs for one case: rows are recordings, columns are
    protocol blocks, plus a TOTAL column, a pass-rate column and a TOTAL row.
    """
    case_df = epochs_df[epochs_df["case"] == case]
    names   = MARKER_NAMES.get(case, {})

    table = pd.crosstab(index=[case_df["subject"], case_df["session"]],
                        columns=case_df["marker"])

    # Re-index against every recording of this case, so a recording that
    # contributed no epochs at all still appears — as a row of zeros — instead of
    # silently vanishing from the table.
    case_log  = log_df[log_df["case"] == case]
    every_rec = pd.MultiIndex.from_frame(case_log[["subject", "session"]])
    table     = table.reindex(every_rec, fill_value=0)

    # Name the columns after the protocol blocks rather than bare marker codes.
    table.columns = [f"{mk} {names.get(mk, '?')}" for mk in table.columns]
    block_cols    = list(table.columns)

    table["TOTAL"] = table[block_cols].sum(axis=1)

    # Pass rate: accepted epochs as a share of the complete windows available.
    # A low value means the recording was noisy; a low TOTAL with a high rate just
    # means it was short.
    table["kept_%"] = (case_log.set_index(["subject", "session"])["kept_pct"]
                               .reindex(table.index).fillna(0.0).values)

    # TOTAL row, with the pass rate pooled across the whole case. Built as its own
    # one-row frame and concatenated, rather than assigned through .loc — a dict
    # assigned to a .loc row is not reliably aligned by column name across pandas
    # versions, and silently produced NaNs here.
    totals = {col: int(table[col].sum()) for col in block_cols + ["TOTAL"]}
    kept, possible = case_log["epochs_kept"].sum(), case_log["epochs_max"].sum()
    totals["kept_%"] = round(100 * kept / possible, 1) if possible else 0.0

    total_row = pd.DataFrame(
        [totals],
        index=pd.MultiIndex.from_tuples([("TOTAL", "")], names=table.index.names),
    )[table.columns]

    return pd.concat([table, total_row])


print("\n" + "=" * 78)
print("  EPOCHS PASSED, BY RECORDING x PROTOCOL BLOCK  (one table per case)")
print("=" * 78)

count_tables = []
for case in sorted(epochs_df["case"].unique()):
    table = epochs_by_recording(case)
    print(f"\n--- Case {case} "
          f"({int(table.loc[('TOTAL', ''), 'TOTAL'])} epochs from "
          f"{len(table) - 1} recording(s)) ---")
    display(table)

    # Keep a long-format copy so every case can be written to one CSV.
    tidy = table.drop(index=("TOTAL", "")).reset_index()
    tidy.insert(0, "case", case)
    count_tables.append(tidy)

# Save the frequency tables next to the other reports for reference.
counts_path = PROJECT_ROOT / "reports" / "epoch_counts_by_recording.csv"
counts_path.parent.mkdir(parents=True, exist_ok=True)
pd.concat(count_tables, ignore_index=True).to_csv(counts_path, index=False)
print(f"\n  Frequency tables saved to {counts_path.relative_to(PROJECT_ROOT)}")

print("\n[STEP 10] Data preparation complete. "
      f"{len(epochs_df):,} epochs are ready in '{OUT_DIR}'.")